# V7ADmerged — the benchmark on five merged AD brain cohorts

Same notebook as **V6ADbio** — same models, hyperparameters, attribution, FDR,
stability selection, enrichment settings and figures — run on a merged matrix
of five GEO cohorts instead of GSE48350 alone. The matrix is built by the
Kaggle notebook **`ad-merged-geo-cohorts`**, attached here as an input. It downloads
every cohort from GEO, applies the project's preprocessing (probe → symbol,
unmapped / multi-gene probes dropped, probes of one gene averaged), keeps genes
measured in all five cohorts, and z-scores each gene within each cohort.

| cohort | platform | region(s) |
|---|---|---|
| GSE48350 | Affymetrix U133 Plus 2.0 | hippocampus, entorhinal, superior frontal, postcentral |
| GSE33000 | Agilent/Rosetta 44K | prefrontal cortex |
| GSE132903 | Illumina HT-12 v4 | middle temporal gyrus |
| GSE118553 | Illumina HT-12 v4 | temporal, entorhinal, frontal cortex |
| GSE84422 | Affymetrix U133A | 15 cortical / hippocampal regions |

Carried over from V6: **donor-grouped splits** (no donor in two partitions),
**age ≥ 60**, **latent geometry on held-out arrays only**, and **every cluster
index scored against the diagnosis**.

New for the merge:
- splits are stratified by **cohort × diagnosis** at the donor level, so every
  partition has every cohort in proportion;
- the label-permuted null permutes diagnoses **across donors within each
  cohort**, which keeps each cohort's AD fraction;
- a **cohort-only** classifier (cohort one-hot → logistic regression) is fit in
  every split, next to the age-only one. Cohorts differ in AD fraction, so
  this shows how much a model could score from cohort identity alone;
- AUC is also reported **per cohort** and **per brain region**.

The comparison cell near the end sets every number against **v5**, so it
records how far each result moved.


In [ ]:
# ======================================================================
# Cell 1 — configuration and data
# ======================================================================
import os, json, warnings, numpy as np, pandas as pd, torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve, confusion_matrix,
                             silhouette_score, adjusted_rand_score,
                             davies_bouldin_score, calinski_harabasz_score)
from scipy import stats
from scipy.stats import rankdata, mannwhitneyu, wilcoxon, ttest_rel

warnings.filterwarnings("ignore")

N_REAL_RUNS = 20          # independent retrained splits, true labels
N_NULL_RUNS = 10          # independent retrained splits, permuted labels
ALPHA       = 0.05        # FDR threshold
IG_STEPS    = 50          # Integrated Gradients path steps
LATENT_DIM  = 8
HIDDEN_DIM  = 256
BETA        = 2.0
GAMMA       = 10.0
MAX_EPOCHS  = 200
PATIENCE    = 20
KL_ANNEAL   = 50
REPRESENTATIVE_SEED = 19  # the split used for single-run illustrations
AGE_MIN     = 60          # V6: age-matched cohort, donors aged >= AGE_MIN only

# Stability-selection windows. The significance test (Mann-Whitney against
# the permuted null) is identical for both methods; only the top-K window is
# widened for the CVAE, whose per-run rankings carry extra estimator variance
# (weight init, batch order, dropout, early-stopping epoch) on top of the
# split variance PLS-DA also sees.
METHOD_PARAMS = {
    "PLS-DA_VIP": {"top_k": 100, "stability_threshold": 0.50},
    "CVAE_IG":    {"top_k": 175, "stability_threshold": 0.40},
}

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

# The published numbers were produced by a session in which CUDA was present but
# its kernels were unusable, so the whole benchmark executed on CPU. PLS-DA is a
# deterministic decomposition and reproduces anywhere; the CVAE does not, because
# dropout masks are drawn from whichever device generator is active. Pinning the
# execution path to CPU is therefore what makes this notebook a reproduction of
# the manuscript rather than a fresh experiment. Set to False to run on GPU --
# expect the CVAE gene list to move, exactly as the paper's Section 4.7 measures.
FORCE_CPU = True


def pick_device():
    """torch.cuda.is_available() is not sufficient on Kaggle: the session can
    be handed a GPU whose compute capability the installed torch build has no
    kernels for, which only surfaces as `no kernel image is available for
    execution on the device` once a real kernel launches. Run a tiny matmul
    and fall back to CPU if it raises."""
    if FORCE_CPU:
        print("FORCE_CPU set; running on CPU as the published run did")
        return torch.device("cpu")
    if torch.cuda.is_available():
        try:
            a = torch.randn(64, 64, device="cuda")
            torch.mm(a, a).sum().item()
            torch.nn.Linear(64, 64).cuda()(a).sum().item()
            return torch.device("cuda")
        except Exception as e:                       # noqa: BLE001
            print(f"CUDA present but unusable ({type(e).__name__}: {e}); using CPU")
    return torch.device("cpu")


device = pick_device()
if device.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 4)
print("device:", device, "| torch", torch.__version__)

# Record the environment rather than leaving it to be inferred from a log. A
# CVAE result is only reproducible against a stated environment, so the versions
# that produced these numbers are part of the output, not part of the metadata.
import platform, sys, importlib


def _ver(mod):
    try:
        return getattr(importlib.import_module(mod), "__version__", "unknown")
    except Exception:                                # noqa: BLE001
        return "not installed"


env = {"python": sys.version.split()[0], "platform": platform.platform(),
       "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "scikit_learn": _ver("sklearn"), "scipy": _ver("scipy"),
       "device": str(device), "force_cpu": FORCE_CPU,
       "cuda_available": torch.cuda.is_available(),
       "cpu_count": os.cpu_count(), "torch_num_threads": torch.get_num_threads()}
pd.DataFrame([{"key": k, "value": v} for k, v in env.items()]).to_csv(
    f"{OUT}/env_report.csv", index=False)
for k, v in env.items():
    print(f"  {k:16} {v}")

import glob
_hits = glob.glob('/kaggle/input/**/merged_expression.csv.gz', recursive=True)
assert _hits, "attach the ad-merged-geo-cohorts notebook output as an input"
MERGED = os.path.dirname(_hits[0])
print("merged data from", MERGED)

df_genes = pd.read_csv(f"{MERGED}/merged_expression.csv.gz")
df_labels = pd.read_csv(f"{MERGED}/merged_labels.csv").rename(columns={'GSM_ID': 'Subject_ID'})
df_meta = pd.read_csv(f"{MERGED}/merged_metadata.csv")
df = (df_genes.merge(df_labels, on='Subject_ID', how='inner', validate='one_to_one')
              .merge(df_meta, on='Subject_ID', how='inner', validate='one_to_one'))
assert len(df) == len(df_genes) == len(df_labels)
assert (df['GROUP'] == df['dx']).all()
print(pd.read_csv(f"{MERGED}/cohort_table.csv").to_string(index=False))
df = df[df['age'] >= AGE_MIN].reset_index(drop=True)      # already applied upstream
print(f"age >= {AGE_MIN}: {len(df)} arrays from {df.donor.nunique()} donors in "
      f"{df.cohort.nunique()} cohorts")
print(df.groupby(['cohort', 'GROUP']).agg(arrays=('Subject_ID', 'size'),
                                          donors=('donor', 'nunique')).to_string())

meta_cols   = ['Subject_ID', 'GROUP', 'donor', 'region', 'age', 'sex', 'dx', 'cohort', 'platform']
gene_cols   = [c for c in df.columns if c not in meta_cols]
donor_ids   = df['donor'].values
regions     = df['region'].values
ages        = df['age'].values.astype(float)
cohorts     = df['cohort'].values
subject_ids = df['Subject_ID'].values
X_raw       = df[gene_cols].values.astype(np.float32)
label_encoder = LabelEncoder()
y_raw       = label_encoder.fit_transform(df['GROUP'].values)
num_classes = len(label_encoder.classes_)
n_genes     = len(gene_cols)
AD_IDX      = int(np.where(label_encoder.classes_ == 'AD')[0][0])

print(f"arrays={X_raw.shape[0]}  donors={len(np.unique(donor_ids))}  genes={n_genes}  "
      f"classes={list(label_encoder.classes_)}")
print(pd.Series(df['GROUP']).value_counts().to_string())

pd.DataFrame({
    "n_subjects": [X_raw.shape[0]], "n_genes": [n_genes],
    "n_AD": [int((y_raw == AD_IDX).sum())],
    "n_control": [int((y_raw != AD_IDX).sum())],
    "class_order": ["|".join(label_encoder.classes_)],
    "AD_index": [AD_IDX],
    "n_donors": [len(np.unique(donor_ids))],
    "n_donors_AD": [len(np.unique(donor_ids[y_raw == AD_IDX]))],
    "n_donors_control": [len(np.unique(donor_ids[y_raw != AD_IDX]))],
    "age_min": [AGE_MIN],
    "n_cohorts": [len(np.unique(cohorts))],
    "age_median_AD": [float(np.median(df.loc[y_raw == AD_IDX].drop_duplicates('donor').age))],
    "age_median_control": [float(np.median(df.loc[y_raw != AD_IDX].drop_duplicates('donor').age))],
}).to_csv(f"{OUT}/cohort_summary.csv", index=False)
df[['Subject_ID', 'GROUP', 'cohort', 'platform', 'donor', 'region', 'age', 'sex']].to_csv(
    f"{OUT}/sample_metadata.csv", index=False)

In [ ]:
# ======================================================================
# Cell 2 — model, losses and shared helpers
# ======================================================================
class GeneExpressionDataset(Dataset):
    def __init__(self, X, y, num_classes):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y).long()
        self.num_classes = num_classes
    def __len__(self):  return self.X.shape[0]
    def __getitem__(self, i):
        y = self.y[i]
        cond = torch.nn.functional.one_hot(y, num_classes=self.num_classes).float()
        return self.X[i], cond, y


class CVAE(nn.Module):
    def __init__(self, input_dim, cond_dim, latent_dim=8, hidden_dim=256, num_classes=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + cond_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2))
        self.mu_layer     = nn.Linear(hidden_dim, latent_dim)
        self.logvar_layer = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, input_dim))
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_classes))

    def encode(self, x, cond):
        h = self.encoder(torch.cat([x, cond], dim=1))
        return self.mu_layer(h), self.logvar_layer(h)

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)

    def decode(self, z, cond):
        return self.decoder(torch.cat([z, cond], dim=1))

    def forward(self, x, cond):
        mu, logvar = self.encode(x, cond)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, cond), mu, logvar, self.classifier(mu)


def cvae_loss(recon_x, x, mu, logvar, logits, y, beta, gamma):
    recon = nn.MSELoss(reduction='mean')(recon_x, x)
    kl    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    ce    = nn.CrossEntropyLoss()(logits, y)
    return recon + beta * kl + gamma * ce


def neutral_cond(n, num_classes, device):
    """Label-agnostic conditioning: uniform over classes, so the encoder is
    given no information about the subject's diagnosis."""
    return torch.full((n, num_classes), 1.0 / num_classes, device=device)


def encode_all(model, X, y, num_classes, device, mode="true", batch_size=256):
    """Return (mu, logvar) for every row of X. mode='true' conditions on the
    observed label; mode='neutral' conditions on the uniform vector."""
    model.eval()
    mus, lvs = [], []
    with torch.no_grad():
        for s in range(0, len(X), batch_size):
            xb = torch.tensor(X[s:s + batch_size], dtype=torch.float32, device=device)
            if mode == "true":
                yb = torch.tensor(y[s:s + batch_size], dtype=torch.long, device=device)
                cb = torch.nn.functional.one_hot(yb, num_classes).float()
            else:
                cb = neutral_cond(len(xb), num_classes, device)
            mu, lv = model.encode(xb, cb)
            mus.append(mu.cpu().numpy()); lvs.append(lv.cpu().numpy())
    return np.vstack(mus), np.vstack(lvs)


def reconstruct_cvae(model, X, y, num_classes, device, mode="true", batch_size=256):
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(X), batch_size):
            xb = torch.tensor(X[s:s + batch_size], dtype=torch.float32, device=device)
            if mode == "true":
                yb = torch.tensor(y[s:s + batch_size], dtype=torch.long, device=device)
                cb = torch.nn.functional.one_hot(yb, num_classes).float()
            else:
                cb = neutral_cond(len(xb), num_classes, device)
            mu, _ = model.encode(xb, cb)
            out.append(model.decode(mu, cb).cpu().numpy())   # decode the mean
    return np.vstack(out)


def per_sample_kl(mu, logvar):
    """KL(q(z|x,c) || N(0,I)) per subject, summed over latent dimensions (nats)."""
    return -0.5 * np.sum(1.0 + logvar - mu ** 2 - np.exp(logvar), axis=1)


def cluster_metrics(Z, y):
    km = KMeans(n_clusters=len(np.unique(y)), n_init=10, random_state=42)
    labels = km.fit_predict(Z)
    return {
        "silhouette":        silhouette_score(Z, y),
        "ari":               adjusted_rand_score(y, labels),
        # V6: scored against the diagnosis, like silhouette (v5 used the
        # k-means partition, which measures k-means rather than diagnosis)
        "davies_bouldin":    davies_bouldin_score(Z, y),
        "calinski_harabasz": calinski_harabasz_score(Z, y),
    }


def inter_intra_ratio(Z, y):
    """Mean between-class centroid distance over mean within-class spread."""
    cents = np.array([Z[y == c].mean(axis=0) for c in np.unique(y)])
    inter = np.linalg.norm(cents[0] - cents[1])
    intra = np.mean([np.linalg.norm(Z[y == c] - cents[i], axis=1).mean()
                     for i, c in enumerate(np.unique(y))])
    return inter / (intra + 1e-12)


def vip_scores(pls):
    """Variable Importance in Projection (Wold et al. 2001): a gene's weight
    aggregated over all components, weighted by the Y-variance each explains."""
    t, w, q = pls.x_scores_, pls.x_weights_, pls.y_loadings_
    p, _ = w.shape
    s = np.sum(t ** 2, axis=0) * np.sum(q ** 2, axis=0)
    w_norm = w / (np.linalg.norm(w, axis=0, keepdims=True) + 1e-12)
    return np.sqrt(p * (w_norm ** 2) @ s / (np.sum(s) + 1e-12))


def bh_fdr(pvals):
    pvals = np.asarray(pvals); n = len(pvals)
    order = np.argsort(pvals)
    q = pvals[order] * n / np.arange(1, n + 1)
    q = np.clip(np.minimum.accumulate(q[::-1])[::-1], 0, 1)
    out = np.empty(n); out[order] = q
    return out

print("helpers defined")

In [ ]:
# ======================================================================
# Cell 3 — Integrated Gradients
# ======================================================================
!pip install captum umap-learn -q
from captum.attr import IntegratedGradients


def ig_attributions(model, X, y, num_classes, device, n_steps=IG_STEPS):
    """Attribute the true-class classifier logit back to genes, relative to a
    zero (= training-mean, post-standardisation) baseline."""
    model.eval()
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    y_t = torch.tensor(y, dtype=torch.long, device=device)
    cond = torch.nn.functional.one_hot(y_t, num_classes).float()

    def forward_fn(x, cond_arg):
        mu, _ = model.encode(x, cond_arg)
        return model.classifier(mu)

    # cond must travel through additional_forward_args: captum expands
    # `inputs` to (batch * n_steps, ...) for the path integral and only
    # applies that expansion to tensors it is told about.
    # internal_batch_size caps that expansion — without it the path integral
    # materialises n_test * n_steps * n_genes floats in one allocation.
    attr = IntegratedGradients(forward_fn).attribute(
        X_t, torch.zeros_like(X_t), target=y_t,
        additional_forward_args=(cond,), n_steps=n_steps,
        internal_batch_size=64)
    return attr.detach().cpu().numpy()


def cvae_latent_saliency(model, X, y, num_classes, device, mode="true", batch_size=128):
    """|d mu_k / d x_j| averaged over subjects — the latent-to-gene map."""
    model.eval()
    accum = np.zeros((model.latent_dim, X.shape[1]), dtype=np.float64)
    n_seen = 0
    for s in range(0, len(X), batch_size):
        xb = torch.tensor(X[s:s + batch_size], dtype=torch.float32,
                          device=device, requires_grad=True)
        if mode == "true":
            yb = torch.tensor(y[s:s + batch_size], dtype=torch.long, device=device)
            cb = torch.nn.functional.one_hot(yb, num_classes).float()
        else:
            cb = neutral_cond(len(xb), num_classes, device)
        mu, _ = model.encode(xb, cb)
        for k in range(model.latent_dim):
            g = torch.autograd.grad(mu[:, k].sum(), xb, retain_graph=(k < model.latent_dim - 1))[0]
            accum[k] += g.abs().sum(dim=0).detach().cpu().numpy()
        n_seen += len(xb)
    return accum / max(n_seen, 1)

print("attribution helpers ready")

In [ ]:
# ======================================================================
# Cell 4 — one split, trained end to end
# ======================================================================
def train_one_split(seed, X_raw, y_raw, permute_labels=False):
    """60/20/20 split over DONORS, stratified by diagnosis, scaler fit on train
    only, PLS-DA and CVAE trained on the identical partition. A donor's arrays
    (up to four brain regions) always land in the same partition."""
    donor_lab = pd.Series(y_raw).groupby(donor_ids).first()      # donor -> label
    donor_coh = pd.Series(cohorts).groupby(donor_ids).first()    # donor -> cohort
    if permute_labels:   # null: permute diagnoses across donors WITHIN each cohort
        rs = np.random.RandomState(seed + 999_000)
        perm = donor_lab.copy()
        for c in sorted(donor_coh.unique()):
            ids = donor_coh.index[donor_coh == c]
            perm.loc[ids] = rs.permutation(donor_lab.loc[ids].values)
        donor_lab = perm
    y_use = donor_lab.loc[donor_ids].values
    strata = donor_coh.astype(str) + "|" + donor_lab.astype(str)   # cohort x diagnosis

    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    d_all = donor_lab.index.values
    d_tr, d_tmp = train_test_split(d_all, test_size=0.4, stratify=strata.loc[d_all].values,
                                   random_state=seed)
    d_va, d_te  = train_test_split(d_tmp, test_size=0.5,
                                   stratify=strata.loc[d_tmp].values, random_state=seed)
    idx_tr = np.where(np.isin(donor_ids, d_tr))[0]
    idx_va = np.where(np.isin(donor_ids, d_va))[0]
    idx_te = np.where(np.isin(donor_ids, d_te))[0]
    assert not (set(donor_ids[idx_tr]) & set(donor_ids[idx_te])), "donor leak train/test"
    assert not (set(donor_ids[idx_tr]) & set(donor_ids[idx_va])), "donor leak train/val"
    assert not (set(donor_ids[idx_va]) & set(donor_ids[idx_te])), "donor leak val/test"
    assert len(idx_tr) + len(idx_va) + len(idx_te) == len(X_raw)

    y_tr, y_va, y_te = y_use[idx_tr], y_use[idx_va], y_use[idx_te]
    scaler  = StandardScaler().fit(X_raw[idx_tr])
    X_tr = scaler.transform(X_raw[idx_tr]).astype(np.float32)
    X_va = scaler.transform(X_raw[idx_va]).astype(np.float32)
    X_te = scaler.transform(X_raw[idx_te]).astype(np.float32)

    # ---- PLS-DA -------------------------------------------------------
    ohe = OneHotEncoder(sparse_output=False)
    plsda = PLSRegression(n_components=LATENT_DIM)
    plsda.fit(X_tr, ohe.fit_transform(y_tr.reshape(-1, 1)))

    # ---- CVAE ---------------------------------------------------------
    # V6: donor-grouped splits give a variable training size; if it leaves a
    # final batch of one array, BatchNorm cannot train on it, so drop only then.
    tr_loader = DataLoader(GeneExpressionDataset(X_tr, y_tr, num_classes), batch_size=32, shuffle=True,
                           drop_last=(len(X_tr) % 32 == 1))
    va_loader = DataLoader(GeneExpressionDataset(X_va, y_va, num_classes), batch_size=32, shuffle=False)

    model = CVAE(X_raw.shape[1], num_classes, LATENT_DIM, HIDDEN_DIM, num_classes).to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=10)

    best_acc, best_state, waited, stopped_at = 0.0, None, 0, MAX_EPOCHS
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        kl_w = min(1.0, epoch / KL_ANNEAL)
        for x, cond, yb in tr_loader:
            x, cond, yb = x.to(device), cond.to(device), yb.to(device)
            opt.zero_grad()
            recon, mu, logvar, logits = model(x, cond)
            cvae_loss(recon, x, mu, logvar, logits, yb, BETA * kl_w, GAMMA).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

        model.eval(); preds, true = [], []
        with torch.no_grad():
            for x, cond, yb in va_loader:
                _, _, _, logits = model(x.to(device), cond.to(device))
                preds.extend(torch.argmax(logits, 1).cpu().numpy()); true.extend(yb.numpy())
        val_acc = accuracy_score(true, preds)
        sched.step(val_acc)

        if val_acc > best_acc + 1e-3:
            best_acc, best_state, waited = val_acc, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            waited += 1
            if waited >= PATIENCE:
                stopped_at = epoch; break

    if best_state is not None:
        model.load_state_dict(best_state)

    return dict(plsda=plsda, model=model, scaler=scaler,
                X_tr=X_tr, X_va=X_va, X_te=X_te, y_tr=y_tr, y_va=y_va, y_te=y_te,
                idx_tr=idx_tr, idx_va=idx_va, idx_te=idx_te,
                best_val_acc=best_acc, epochs_run=stopped_at)

print("split trainer ready")

In [ ]:
# ======================================================================
# Cell 5 — 20 real splits: classification, geometry, reconstruction, KL
# ======================================================================
BASE_FPR = np.linspace(0, 1, 101)
BASE_REC = np.linspace(0, 1, 101)

run_rows, roc_rows, pr_rows = [], [], []
oof_recon, oof_unc, oof_pred = [], [], []
cm_sum = {"pls": np.zeros((2, 2)), "cvae_true": np.zeros((2, 2)), "cvae_neutral": np.zeros((2, 2))}
vip_real = np.zeros((N_REAL_RUNS, n_genes))
ig_real  = np.zeros((N_REAL_RUNS, n_genes))
representative = None

for r in range(N_REAL_RUNS):
    S = train_one_split(r, X_raw, y_raw)
    plsda, model = S["plsda"], S["model"]
    X_tr, X_te, y_tr, y_te = S["X_tr"], S["X_te"], S["y_tr"], S["y_te"]

    # ---- PLS-DA classification ---------------------------------------
    y_ad = (y_te == AD_IDX).astype(int)          # 1 = AD, the positive class
    Ztr_p, Zte_p = plsda.transform(X_tr), plsda.transform(X_te)
    clf_p = LogisticRegression(max_iter=1000).fit(Ztr_p, y_tr)
    prob_p = clf_p.predict_proba(Zte_p)[:, list(clf_p.classes_).index(AD_IDX)]
    pred_p = clf_p.predict(Zte_p)

    # ---- CVAE classification, both conditioning arms ------------------
    res = {}
    for mode in ("true", "neutral"):
        mu_tr, _        = encode_all(model, X_tr, y_tr, num_classes, device, mode)
        mu_te, lv_te    = encode_all(model, X_te, y_te, num_classes, device, mode)
        clf_c = LogisticRegression(max_iter=1000).fit(mu_tr, y_tr)
        res[mode] = dict(mu_tr=mu_tr, mu_te=mu_te, lv_te=lv_te,
                         prob=clf_c.predict_proba(mu_te)[:, list(clf_c.classes_).index(AD_IDX)],
                         pred=clf_c.predict(mu_te))
        with torch.no_grad():
            logits = model.classifier(torch.tensor(mu_te, dtype=torch.float32, device=device))
            res[mode]["head_prob"] = torch.softmax(logits, 1)[:, AD_IDX].cpu().numpy()
            res[mode]["head_pred"] = torch.argmax(logits, 1).cpu().numpy()

    # ---- V6: age-only reference classifier ---------------------------
    clf_a = LogisticRegression(max_iter=1000).fit(ages[S["idx_tr"]].reshape(-1, 1), y_tr)
    prob_a = clf_a.predict_proba(ages[S["idx_te"]].reshape(-1, 1))[:, list(clf_a.classes_).index(AD_IDX)]

    # ---- V7: cohort-only reference classifier --------------------------
    coh_levels = sorted(np.unique(cohorts))
    C_tr = (cohorts[S["idx_tr"]][:, None] == np.array(coh_levels)[None, :]).astype(float)
    C_te = (cohorts[S["idx_te"]][:, None] == np.array(coh_levels)[None, :]).astype(float)
    clf_k = LogisticRegression(max_iter=1000).fit(C_tr, y_tr)
    prob_k = clf_k.predict_proba(C_te)[:, list(clf_k.classes_).index(AD_IDX)]

    # ---- per-run metric row ------------------------------------------
    row = {"run": r + 1, "seed": r, "n_test": len(y_te),
           "n_test_donors": len(np.unique(donor_ids[S["idx_te"]])),
           "auc_age_only": roc_auc_score(y_ad, prob_a),
           "auc_cohort_only": roc_auc_score(y_ad, prob_k),
           "best_val_acc": S["best_val_acc"], "epochs_run": S["epochs_run"],
           "acc_pls": accuracy_score(y_te, pred_p), "auc_pls": roc_auc_score(y_ad, prob_p),
           "ap_pls": average_precision_score(y_ad, prob_p)}
    for mode in ("true", "neutral"):
        row[f"acc_cvae_lat_{mode}"]  = accuracy_score(y_te, res[mode]["pred"])
        row[f"auc_cvae_lat_{mode}"]  = roc_auc_score(y_ad, res[mode]["prob"])
        row[f"ap_cvae_lat_{mode}"]   = average_precision_score(y_ad, res[mode]["prob"])
        row[f"acc_cvae_head_{mode}"] = accuracy_score(y_te, res[mode]["head_pred"])
        row[f"auc_cvae_head_{mode}"] = roc_auc_score(y_ad, res[mode]["head_prob"])

    # ---- V6: donor-level AUC (mean P(AD) over each held-out donor's arrays)
    d_te_ids = donor_ids[S["idx_te"]]
    d_y = pd.Series(y_ad).groupby(d_te_ids).first()
    for tag, prob in (("pls", prob_p), ("cvae_lat_true", res["true"]["prob"]),
                      ("cvae_lat_neutral", res["neutral"]["prob"])):
        d_p = pd.Series(prob).groupby(d_te_ids).mean().loc[d_y.index]
        row[f"auc_{tag}_donor"] = roc_auc_score(d_y.values, d_p.values)

    # ---- latent geometry, V6: held-out arrays only ---------------------
    y_full = y_te
    geo_sets = {"pls": Zte_p,
                "cvae_true":    res["true"]["mu_te"],
                "cvae_neutral": res["neutral"]["mu_te"]}
    for name, Z in geo_sets.items():
        m = cluster_metrics(Z, y_full)
        row[f"sil_{name}"] = m["silhouette"];        row[f"ari_{name}"] = m["ari"]
        row[f"db_{name}"]  = m["davies_bouldin"];    row[f"ch_{name}"]  = m["calinski_harabasz"]
        row[f"iir_{name}"] = inter_intra_ratio(Z, y_full)

    # ---- reconstruction, held-out subjects ---------------------------
    recon_p = Zte_p @ plsda.x_loadings_.T
    mse_p   = np.mean((X_te - recon_p) ** 2, axis=1)
    mse_c   = {mode: np.mean((X_te - reconstruct_cvae(model, X_te, y_te, num_classes, device, mode)) ** 2, axis=1)
               for mode in ("true", "neutral")}
    row["mse_pls_mean"]           = mse_p.mean()
    row["mse_cvae_true_mean"]     = mse_c["true"].mean()
    row["mse_cvae_neutral_mean"]  = mse_c["neutral"].mean()

    # ---- per-subject out-of-fold records ------------------------------
    for i, gidx in enumerate(S["idx_te"]):
        oof_recon.append({"run": r + 1, "subject_id": subject_ids[gidx],
                          "group": label_encoder.classes_[y_te[i]],
                          "mse_pls": mse_p[i],
                          "mse_cvae_true": mse_c["true"][i],
                          "mse_cvae_neutral": mse_c["neutral"][i]})
        # every classifier's posterior P(AD) for this held-out subject, so the
        # confidence landscapes are drawn from out-of-fold predictions only
        prow = {"run": r + 1, "subject_id": subject_ids[gidx],
                "group": label_encoder.classes_[y_te[i]],
                "donor": donor_ids[gidx], "region": regions[gidx], "age": ages[gidx],
                "cohort": cohorts[gidx],
                "p_pls": float(prob_p[i])}
        for mode in ("true", "neutral"):
            prow[f"p_cvae_lat_{mode}"]  = float(res[mode]["prob"][i])
            prow[f"p_cvae_head_{mode}"] = float(res[mode]["head_prob"][i])
        oof_pred.append(prow)

        rec = {"run": r + 1, "subject_id": subject_ids[gidx],
               "group": label_encoder.classes_[y_te[i]]}
        for mode in ("true", "neutral"):
            mu_i, lv_i = res[mode]["mu_te"][i], res[mode]["lv_te"][i]
            rec[f"kl_{mode}"]       = float(-0.5 * np.sum(1 + lv_i - mu_i ** 2 - np.exp(lv_i)))
            rec[f"kl_perdim_{mode}"] = rec[f"kl_{mode}"] / LATENT_DIM
            rec[f"meanvar_{mode}"]   = float(np.mean(np.exp(lv_i)))
        oof_unc.append(rec)

    # ---- curves and confusion matrices -------------------------------
    for tag, prob, pred in (("pls", prob_p, pred_p),
                            ("cvae_true", res["true"]["prob"], res["true"]["pred"]),
                            ("cvae_neutral", res["neutral"]["prob"], res["neutral"]["pred"])):
        fpr, tpr, _ = roc_curve(y_ad, prob)
        roc_rows.append({"run": r + 1, "model": tag,
                         **{f"tpr_{i}": v for i, v in enumerate(np.interp(BASE_FPR, fpr, tpr))}})
        prec, rec_, _ = precision_recall_curve(y_ad, prob)
        o = np.argsort(rec_)
        pr_rows.append({"run": r + 1, "model": tag,
                        **{f"prec_{i}": v for i, v in enumerate(np.interp(BASE_REC, rec_[o], prec[o]))}})
        # rows and columns ordered [Control, AD] so tn/fp/fn/tp read with AD positive
        cm_sum[tag] += confusion_matrix(y_te, pred, labels=[1 - AD_IDX, AD_IDX])

    # ---- attribution --------------------------------------------------
    vip_real[r] = vip_scores(plsda)
    ig_real[r]  = np.abs(ig_attributions(model, X_te, y_te, num_classes, device)).mean(axis=0)

    if r == REPRESENTATIVE_SEED:
        representative = dict(S=S, res=res, Ztr_p=Ztr_p, Zte_p=Zte_p,
                              mse_p=mse_p, mse_c=mse_c, recon_p=recon_p)

    run_rows.append(row)
    print(f"run {r+1:2d}/{N_REAL_RUNS}  acc_pls={row['acc_pls']:.3f} "
          f"auc_pls={row['auc_pls']:.3f} | acc_cvae={row['acc_cvae_lat_true']:.3f} "
          f"auc_cvae={row['auc_cvae_lat_true']:.3f} | neutral auc={row['auc_cvae_lat_neutral']:.3f} "
          f"| age-only auc={row['auc_age_only']:.3f} | cohort-only auc={row['auc_cohort_only']:.3f} "
          f"| test donors={row['n_test_donors']}")

runs = pd.DataFrame(run_rows)
runs.to_csv(f"{OUT}/run_metrics.csv", index=False)
pd.DataFrame(roc_rows).to_csv(f"{OUT}/roc_curves_raw.csv", index=False)
pd.DataFrame(pr_rows).to_csv(f"{OUT}/pr_curves_raw.csv", index=False)
pd.DataFrame(oof_recon).to_csv(f"{OUT}/oof_reconstruction.csv", index=False)
pd.DataFrame(oof_unc).to_csv(f"{OUT}/oof_uncertainty.csv", index=False)
pd.DataFrame(oof_pred).to_csv(f"{OUT}/oof_predictions.csv", index=False)
pd.DataFrame([{"model": k, "tn": v[0, 0], "fp": v[0, 1], "fn": v[1, 0], "tp": v[1, 1]}
              for k, v in cm_sum.items()]).to_csv(f"{OUT}/confusion_matrices.csv", index=False)
print("\nreal runs complete")

In [ ]:
# ======================================================================
# Cell 6 — 10 label-permuted null splits
# ======================================================================
vip_null = np.zeros((N_NULL_RUNS, n_genes))
ig_null  = np.zeros((N_NULL_RUNS, n_genes))

for r in range(N_NULL_RUNS):
    S = train_one_split(10_000 + r, X_raw, y_raw, permute_labels=True)
    vip_null[r] = vip_scores(S["plsda"])
    ig_null[r]  = np.abs(ig_attributions(S["model"], S["X_te"], S["y_te"],
                                         num_classes, device)).mean(axis=0)
    print(f"null run {r+1}/{N_NULL_RUNS}")

np.save(f"{OUT}/vip_real.npy", vip_real); np.save(f"{OUT}/vip_null.npy", vip_null)
np.save(f"{OUT}/ig_real.npy", ig_real);   np.save(f"{OUT}/ig_null.npy", ig_null)
print("null runs complete")

In [ ]:
# ======================================================================
# Cell 7 — gene significance, stability, direction
# ======================================================================
def summarize(method, real, null, genes, top_k, stability_threshold):
    """Rank-based test against the permuted null.

    Magnitudes cannot be compared directly: mean(VIP^2) equals 1 for any
    fitted PLS model by construction, and a network trained on permuted
    labels can still produce large IG attributions by memorising noise.
    Ranking within each run sidesteps both — a gene consistently top-ranked
    across real runs but randomly ranked across null runs is real signal.
    """
    n_real, n_g = real.shape
    real_ranks = np.array([rankdata(-real[i]) for i in range(n_real)])
    null_ranks = np.array([rankdata(-null[i]) for i in range(null.shape[0])])

    pvals = np.ones(n_g)
    for j in range(n_g):
        try:
            pvals[j] = mannwhitneyu(real_ranks[:, j], null_ranks[:, j], alternative='less')[1]
        except ValueError:
            pvals[j] = 1.0
    qvals = bh_fdr(pvals)

    flags = np.zeros_like(real, dtype=bool)
    for i in range(n_real):
        flags[i, np.argsort(real[i])[::-1][:top_k]] = True
    stability = flags.mean(axis=0)

    out = pd.DataFrame({
        "gene": genes, "mean_score": real.mean(axis=0), "std_score": real.std(axis=0),
        "p_value": pvals, "fdr_q": qvals, "stability_freq": stability,
        "significant": (qvals < ALPHA) & (stability >= stability_threshold),
    }).sort_values("mean_score", ascending=False).reset_index(drop=True)
    out.insert(0, "method", method)
    return out


pls_stats  = summarize("PLS-DA_VIP", vip_real, vip_null, gene_cols, **METHOD_PARAMS["PLS-DA_VIP"])
cvae_stats = summarize("CVAE_IG",    ig_real,  ig_null,  gene_cols, **METHOD_PARAMS["CVAE_IG"])

# ---- differential expression on standardised whole-cohort expression ----
X_z = StandardScaler().fit_transform(X_raw)
ad_mask = (y_raw == AD_IDX)
logfc = X_z[ad_mask].mean(axis=0) - X_z[~ad_mask].mean(axis=0)
de_p  = np.array([stats.mannwhitneyu(X_z[ad_mask, j], X_z[~ad_mask, j],
                                     alternative='two-sided')[1] for j in range(n_genes)])
de = pd.DataFrame({"gene": gene_cols, "logFC": logfc, "DE_pvalue": de_p,
                   "DE_qvalue": bh_fdr(de_p),
                   "mean_z_AD": X_z[ad_mask].mean(axis=0),
                   "mean_z_control": X_z[~ad_mask].mean(axis=0)})
de.to_csv(f"{OUT}/gene_de_stats.csv", index=False)


def add_direction(stat_df):
    m = stat_df.merge(de, on="gene", how="left")
    m["direction"] = "none"
    m.loc[(m.DE_pvalue < 0.10) & (m.logFC > 0), "direction"] = "up"
    m.loc[(m.DE_pvalue < 0.10) & (m.logFC < 0), "direction"] = "down"
    return m


pls_stats  = add_direction(pls_stats)
cvae_stats = add_direction(cvae_stats)
pls_stats.to_csv(f"{OUT}/PLSDA_gene_significance.csv", index=False)
cvae_stats.to_csv(f"{OUT}/CVAE_gene_significance.csv", index=False)

pls_sig  = set(pls_stats.loc[pls_stats.significant, "gene"])
cvae_sig = set(cvae_stats.loc[cvae_stats.significant, "gene"])
overlap  = pls_sig & cvae_sig
jaccard  = len(overlap) / max(len(pls_sig | cvae_sig), 1)
rho, rho_p = stats.spearmanr(vip_real.mean(axis=0), ig_real.mean(axis=0))

(pls_stats[pls_stats.gene.isin(overlap)][["gene", "logFC", "DE_pvalue", "direction"]]
 .drop_duplicates("gene").to_csv(f"{OUT}/consensus_significant_genes.csv", index=False))

for name, dfx, d in [("PLSDA_upregulated", pls_stats, "up"), ("PLSDA_downregulated", pls_stats, "down"),
                     ("CVAE_upregulated", cvae_stats, "up"), ("CVAE_downregulated", cvae_stats, "down")]:
    sel = dfx[dfx.significant & (dfx.direction == d)]
    sel.to_csv(f"{OUT}/{name}_genes.csv", index=False)
    print(f"{name}: {len(sel)}")

print(f"\nPLS-DA significant: {len(pls_sig)}   CVAE significant: {len(cvae_sig)}")
print(f"overlap: {len(overlap)} -> {sorted(overlap)}   Jaccard={jaccard:.4f}")
print(f"Spearman rho between mean VIP and mean IG: {rho:.4f} (p={rho_p:.3e})")

In [ ]:
# ======================================================================
# Cell 8 — latent structure of the representative split
# ======================================================================
S   = representative["S"]
res = representative["res"]
model, plsda = S["model"], S["plsda"]
X_full_rep = np.vstack([S["X_tr"], S["X_te"]])
y_full_rep = np.concatenate([S["y_tr"], S["y_te"]])

# PLS-DA latent-to-gene map: |corr(latent score, gene)|
Z_pls_full = plsda.transform(X_full_rep)
cov = (((Z_pls_full - Z_pls_full.mean(0)).T @ (X_full_rep - X_full_rep.mean(0)))
       / (len(X_full_rep) - 1))
plsd_weights = np.abs(cov / np.outer(np.maximum(Z_pls_full.std(0, ddof=1), 1e-12),
                                     np.maximum(X_full_rep.std(0, ddof=1), 1e-12)))

snr_rows, embed_rows = [], []
for tag, W, prefix in [("PLS-DA", plsd_weights, "LV"),
                       ("CVAE_true",    cvae_latent_saliency(model, X_full_rep, y_full_rep,
                                                             num_classes, device, "true"), "Z"),
                       ("CVAE_neutral", cvae_latent_saliency(model, X_full_rep, y_full_rep,
                                                             num_classes, device, "neutral"), "Z")]:
    sig = pls_sig if tag == "PLS-DA" else cvae_sig
    mask = np.array([g in sig for g in gene_cols])
    for k in range(min(LATENT_DIM, W.shape[0])):
        w = np.abs(W[k])
        snr_rows.append({"model": tag, "latent": f"{prefix}{k+1}",
                         "signal": w[mask].mean(), "noise": w[~mask].mean(),
                         "snr": w[mask].mean() / (w[~mask].mean() + 1e-12)})
    np.save(f"{OUT}/latent_gene_weights_{tag}.npy", W[:LATENT_DIM])

pd.DataFrame(snr_rows).to_csv(f"{OUT}/latent_snr.csv", index=False)

# 2-D embeddings for the geometry figure. UMAP is optional: it is not part of
# the benchmark, only of the display, so a missing package must not fail the run.
try:
    import umap
    HAVE_UMAP = True
except Exception as e:
    print(f"umap-learn unavailable ({e}); PCA and t-SNE only")
    HAVE_UMAP = False

emb = {"PLS-DA": Z_pls_full,
       "CVAE_true":    np.vstack([res["true"]["mu_tr"],    res["true"]["mu_te"]]),
       "CVAE_neutral": np.vstack([res["neutral"]["mu_tr"], res["neutral"]["mu_te"]])}
is_test = np.zeros(len(y_full_rep), bool)
is_test[len(S["y_tr"]):] = True

coord_rows = []
for tag, Z in emb.items():
    p2 = PCA(n_components=2, random_state=42).fit_transform(Z)
    t2 = TSNE(n_components=2, random_state=42,
              perplexity=min(30, max(5, len(Z) // 4)), init="pca").fit_transform(Z)
    if HAVE_UMAP:
        u2 = umap.UMAP(n_components=2, random_state=42,
                       n_neighbors=min(15, len(Z) - 1), min_dist=0.1).fit_transform(Z)
    else:
        u2 = np.full((len(Z), 2), np.nan)
    for i in range(len(Z)):
        embed_rows.append({"model": tag, "group": label_encoder.classes_[y_full_rep[i]],
                           "split": "test" if is_test[i] else "train",
                           "pca1": p2[i, 0], "pca2": p2[i, 1],
                           "tsne1": t2[i, 0], "tsne2": t2[i, 1],
                           "umap1": u2[i, 0], "umap2": u2[i, 1]})
        coord_rows.append({"model": tag, "group": label_encoder.classes_[y_full_rep[i]],
                           "split": "test" if is_test[i] else "train",
                           **{f"z{k+1}": Z[i, k] for k in range(Z.shape[1])}})
pd.DataFrame(embed_rows).to_csv(f"{OUT}/latent_embeddings.csv", index=False)
pd.DataFrame(coord_rows).to_csv(f"{OUT}/latent_coords_representative.csv", index=False)

# representative-split reconstruction scatter (subsampled for plotting)
rng = np.random.RandomState(0)
pick = rng.choice(S["X_te"].size, size=min(60000, S["X_te"].size), replace=False)
recon_c_true = reconstruct_cvae(model, S["X_te"], S["y_te"], num_classes, device, "true")
pd.DataFrame({"observed": S["X_te"].ravel()[pick],
              "recon_pls": representative["recon_p"].ravel()[pick],
              "recon_cvae": recon_c_true.ravel()[pick]}).to_csv(
    f"{OUT}/recon_scatter_representative.csv", index=False)
print("latent structure written")

In [ ]:
# ======================================================================
# Cell 9 — every scalar the article quotes
# ======================================================================
def paired(a, b):
    t, pt = ttest_rel(a, b)
    try:    w, pw = wilcoxon(a, b)
    except ValueError: w, pw = np.nan, np.nan
    d = np.mean(a - b) / (np.std(a - b, ddof=1) + 1e-12)
    return t, pt, w, pw, d


scal = []
def put(k, v, note=""):
    scal.append({"key": k, "value": v, "note": note})

for m in ["acc_pls", "auc_pls", "ap_pls",
          "acc_cvae_lat_true", "auc_cvae_lat_true", "ap_cvae_lat_true",
          "acc_cvae_lat_neutral", "auc_cvae_lat_neutral", "ap_cvae_lat_neutral",
          "acc_cvae_head_true", "auc_cvae_head_true",
          "acc_cvae_head_neutral", "auc_cvae_head_neutral",
          "sil_pls", "ari_pls", "db_pls", "ch_pls", "iir_pls",
          "sil_cvae_true", "ari_cvae_true", "db_cvae_true", "ch_cvae_true", "iir_cvae_true",
          "sil_cvae_neutral", "ari_cvae_neutral", "db_cvae_neutral", "ch_cvae_neutral",
          "iir_cvae_neutral", "mse_pls_mean", "mse_cvae_true_mean", "mse_cvae_neutral_mean"]:
    put(f"{m}_mean", runs[m].mean()); put(f"{m}_sd", runs[m].std())

for label, a, b in [("acc_cvae_true_vs_pls", runs.acc_cvae_lat_true, runs.acc_pls),
                    ("auc_cvae_true_vs_pls", runs.auc_cvae_lat_true, runs.auc_pls),
                    ("acc_cvae_neutral_vs_pls", runs.acc_cvae_lat_neutral, runs.acc_pls),
                    ("auc_cvae_neutral_vs_pls", runs.auc_cvae_lat_neutral, runs.auc_pls),
                    ("acc_cvae_true_vs_neutral", runs.acc_cvae_lat_true, runs.acc_cvae_lat_neutral),
                    ("auc_cvae_true_vs_neutral", runs.auc_cvae_lat_true, runs.auc_cvae_lat_neutral)]:
    t, pt, w, pw, d = paired(a.values, b.values)
    put(f"{label}_t", t); put(f"{label}_p", pt); put(f"{label}_cohen_dz", d)

# reconstruction, pooled over all out-of-fold subjects
orec = pd.DataFrame(oof_recon)
for mode in ("true", "neutral"):
    a, b = orec[f"mse_cvae_{mode}"].values, orec["mse_pls"].values
    t, pt, w, pw, d = paired(a, b)
    rb = 2 * np.mean(a > b) - 1
    put(f"recon_cvae_{mode}_vs_pls_wilcoxon_p", pw)
    put(f"recon_cvae_{mode}_vs_pls_ttest_p", pt)
    put(f"recon_cvae_{mode}_vs_pls_cohen_dz", d)
    put(f"recon_cvae_{mode}_vs_pls_rank_biserial", rb)
put("recon_n_oof_samples", len(orec))
for g in orec.group.unique():
    sub = orec[orec.group == g]
    put(f"recon_mse_pls_{g}_mean", sub.mse_pls.mean())
    put(f"recon_mse_pls_{g}_sd", sub.mse_pls.std())
    put(f"recon_mse_cvae_true_{g}_mean", sub.mse_cvae_true.mean())
    put(f"recon_mse_cvae_true_{g}_sd", sub.mse_cvae_true.std())
    put(f"recon_n_{g}", len(sub))

# within a model, does reconstruction error depend on diagnosis? A model that
# rebuilt one group better than the other would confound every downstream claim.
for col, tag in [("mse_pls", "pls"), ("mse_cvae_true", "cvaetrue"),
                 ("mse_cvae_neutral", "cvaeneutral")]:
    a = orec.loc[orec.group == "AD", col].values
    b = orec.loc[orec.group != "AD", col].values
    put(f"recon_group_{tag}_mannwhitney_p", mannwhitneyu(a, b, alternative='two-sided')[1])

# confusion-matrix operating point, summed over the 20 held-out folds
cm_df = pd.DataFrame([{"model": k, "tn": v[0, 0], "fp": v[0, 1], "fn": v[1, 0], "tp": v[1, 1]}
                      for k, v in cm_sum.items()])
for _, cr in cm_df.iterrows():
    tag = cr["model"].replace("_", "")
    tn, fp, fn, tp = cr.tn, cr.fp, cr.fn, cr.tp
    put(f"cm_{tag}_sensitivity", tp / max(tp + fn, 1))
    put(f"cm_{tag}_specificity", tn / max(tn + fp, 1))
    put(f"cm_{tag}_precision", tp / max(tp + fp, 1))
    put(f"cm_{tag}_balanced_accuracy", 0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1)))

# out-of-fold posterior confidence: how far apart are the two classes' P(AD)?
opred = pd.DataFrame(oof_pred)
for col, tag in [("p_pls", "pls"), ("p_cvae_lat_true", "cvaelat"),
                 ("p_cvae_head_true", "cvaehead")]:
    a = opred.loc[opred.group == "AD", col].values
    b = opred.loc[opred.group != "AD", col].values
    put(f"prob_{tag}_AD_mean", a.mean())
    put(f"prob_{tag}_control_mean", b.mean())
    put(f"prob_{tag}_gap", a.mean() - b.mean())
    put(f"prob_{tag}_mannwhitney_p", mannwhitneyu(a, b, alternative='two-sided')[1])
    # fraction of subjects the model places within 0.1 of the decision threshold
    put(f"prob_{tag}_frac_ambiguous",
        float(np.mean(np.abs(opred[col].values - 0.5) < 0.1)))

# out-of-fold uncertainty: AD vs control, both conditioning arms
ounc = pd.DataFrame(oof_unc)
for mode in ("true", "neutral"):
    for col in (f"kl_{mode}", f"meanvar_{mode}"):
        ad  = ounc.loc[ounc.group == "AD", col].values
        ctl = ounc.loc[ounc.group != "AD", col].values
        u, p = mannwhitneyu(ad, ctl, alternative='two-sided')
        put(f"{col}_AD_mean", ad.mean());   put(f"{col}_AD_median", np.median(ad))
        put(f"{col}_control_mean", ctl.mean()); put(f"{col}_control_median", np.median(ctl))
        put(f"{col}_mannwhitney_p", p)
        put(f"{col}_pct_increase_AD", 100 * (ad.mean() - ctl.mean()) / abs(ctl.mean()))
put("unc_n_AD", int((ounc.group == "AD").sum()))
put("unc_n_control", int((ounc.group != "AD").sum()))

put("n_sig_plsda", len(pls_sig)); put("n_sig_cvae", len(cvae_sig))
put("n_overlap", len(overlap));   put("jaccard", jaccard)
put("consensus_genes", "|".join(sorted(overlap)))
put("spearman_rho_vip_ig", rho); put("spearman_p_vip_ig", rho_p)
put("n_total_discovered", len(pls_sig | cvae_sig))
put("n_plsda_only", len(pls_sig - cvae_sig)); put("n_cvae_only", len(cvae_sig - pls_sig))
put("n_not_significant", n_genes - len(pls_sig | cvae_sig))
put("mean_stability_plsda", pls_stats.loc[pls_stats.significant, "stability_freq"].mean())
put("mean_stability_cvae", cvae_stats.loc[cvae_stats.significant, "stability_freq"].mean())
put("n_genes", n_genes); put("n_subjects", len(X_raw))

# ---- V6 additions -------------------------------------------------------
put("n_AD", int((y_raw == AD_IDX).sum()))
put("n_donors", len(np.unique(donor_ids)))
put("n_donors_AD", len(np.unique(donor_ids[y_raw == AD_IDX])))
put("n_donors_control", len(np.unique(donor_ids[y_raw != AD_IDX])))
put("age_min", AGE_MIN)
put("mean_test_donors", runs.n_test_donors.mean())
put("n_cohorts", len(np.unique(cohorts)))
for m in ["auc_age_only", "auc_cohort_only", "auc_pls_donor", "auc_cvae_lat_true_donor",
          "auc_cvae_lat_neutral_donor"]:
    put(f"{m}_mean", runs[m].mean()); put(f"{m}_sd", runs[m].std())
# pooled out-of-fold AUC within each cohort
for coh in sorted(opred.cohort.unique()):
    sub = opred[opred.cohort == coh]
    yy = (sub.group == "AD").astype(int).values
    put(f"cohort_{coh.lower()}_n", len(sub))
    if yy.min() != yy.max():
        put(f"cohort_{coh.lower()}_auc_pls", roc_auc_score(yy, sub.p_pls))
        put(f"cohort_{coh.lower()}_auc_cvae", roc_auc_score(yy, sub.p_cvae_lat_true))
# pooled out-of-fold AUC within each brain region
for reg in sorted(opred.region.unique()):
    sub = opred[opred.region == reg]
    rtag = reg.replace(" ", "_")
    yy = (sub.group == "AD").astype(int).values
    put(f"region_{rtag}_n", len(sub))
    if yy.min() != yy.max():
        put(f"region_{rtag}_auc_pls", roc_auc_score(yy, sub.p_pls))
        put(f"region_{rtag}_auc_cvae", roc_auc_score(yy, sub.p_cvae_lat_true))
put("mean_epochs_run", runs.epochs_run.mean())

pd.DataFrame(scal).to_csv(f"{OUT}/summary_scalars.csv", index=False)
print(pd.DataFrame(scal).to_string(max_rows=400))
print("\nAll CSVs written to", OUT)
print(sorted(os.listdir(OUT)))

In [ ]:
# ======================================================================
# Cell 10 — functional enrichment (g:Profiler), same background for all sets
# ======================================================================
import requests

GPROFILER = "https://biit.cs.ut.ee/gprofiler/api/gost/profile/"
SETS = {
    "plsda_up":   list(pd.read_csv(f"{OUT}/PLSDA_upregulated_genes.csv").gene),
    "plsda_down": list(pd.read_csv(f"{OUT}/PLSDA_downregulated_genes.csv").gene),
    "cvae_up":    list(pd.read_csv(f"{OUT}/CVAE_upregulated_genes.csv").gene),
    "cvae_down":  list(pd.read_csv(f"{OUT}/CVAE_downregulated_genes.csv").gene),
}

os.makedirs(f"{OUT}/enrichment", exist_ok=True)
enr_counts = []
for name, genes in SETS.items():
    EMPTY = pd.DataFrame(columns=["source", "term_name", "term_id", "adjusted_p_value",
                                  "term_size", "query_size", "intersection_size",
                                  "intersections", "negative_log10_of_adjusted_p_value"])
    if not genes:
        print(f"{name}: empty gene set, skipped")
        EMPTY.to_csv(f"{OUT}/enrichment/{name}.csv", index=False)   # V6: keep the file
        enr_counts.append({"set": name, "n_genes": 0, "n_terms": 0})
        continue
    # the full preprocessed gene universe is the background, so enrichment is
    # not inflated by g:Profiler's default whole-genome domain
    payload = {"organism": "hsapiens", "query": genes, "sources": [],
               "user_threshold": 0.05, "significance_threshold_method": "fdr",
               "background": gene_cols, "domain_scope": "custom",
               "no_evidences": False}
    try:
        r = requests.post(GPROFILER, json=payload,
                          headers={"User-Agent": "ad-latent-benchmark"}, timeout=180)
        r.raise_for_status()
        body = r.json()
        res = pd.DataFrame(body.get("result", []))
    except Exception as e:                        # noqa: BLE001
        print(f"{name}: g:Profiler request failed ({type(e).__name__}: {e})")
        enr_counts.append({"set": name, "n_genes": len(genes), "n_terms": -1})
        continue

    if len(res):
        # `intersections` is one entry per *converted* query gene, holding that
        # gene's evidence codes and empty when the gene is not in the term --
        # it is not a gene list. Recover symbols by position against the query
        # order g:Profiler reports back in the metadata.
        try:
            qmeta = body["meta"]["genes_metadata"]["query"]
            order = qmeta[next(iter(qmeta))]["ensgs"]
            mapping = qmeta[next(iter(qmeta))].get("mapping", {})
            ensg2sym = {e: s for s, es in mapping.items()
                        for e in (es if isinstance(es, list) else [es])}
            qsyms = [ensg2sym.get(e, e) for e in order]
        except Exception:                          # noqa: BLE001
            qsyms = genes

        def hit_genes(v):
            if not isinstance(v, list):
                return ""
            return ",".join(qsyms[i] for i, ev in enumerate(v)
                            if ev and i < len(qsyms))

        res["hit_genes"] = res.intersections.apply(hit_genes) if "intersections" in res else ""
        keep = [c for c in ["source", "name", "native", "p_value", "term_size",
                            "query_size", "intersection_size", "hit_genes"] if c in res]
        res = res[keep].rename(columns={"name": "term_name", "native": "term_id",
                                        "p_value": "adjusted_p_value",
                                        "hit_genes": "intersections"})
        res["negative_log10_of_adjusted_p_value"] = -np.log10(res.adjusted_p_value)
        res = res.sort_values("adjusted_p_value")
    res.to_csv(f"{OUT}/enrichment/{name}.csv", index=False)
    enr_counts.append({"set": name, "n_genes": len(genes), "n_terms": len(res)})
    print(f"{name}: {len(genes)} genes -> {len(res)} enriched terms")

pd.DataFrame(enr_counts).to_csv(f"{OUT}/enrichment_counts.csv", index=False)
print("\nenrichment written to", f"{OUT}/enrichment")

---

# Reproducibility, the paper's numbers, and a check against the manuscript

Three steps close the loop between this notebook and the manuscript:

1. **Gene-list overlap with v5.** In V6 the `repro_*` numbers compare this
   run's gene lists with **v5's** (the published run): how many genes survive
   donor-grouped, age-matched resampling, and where v5's genes now rank.
2. **`values.tex`** is the actual file `main.tex` reads. Every number quoted in
   the paper is one of these macros, so nothing in the text can drift from the
   analysis above.
3. **The comparison with v5** sets every scalar this run produced against v5's
   value. Drift is expected here; the table records how much each number moved.

All three run the paper's own scripts -- `scripts/make_repro.py`,
`scripts/make_macros.py` and `kaggle/reproduction_check.py` -- pasted in
unchanged apart from their import lines.

In [ ]:
# Point the paper's scripts at this notebook's output directory.
import os
os.environ["PAPER_DATA"] = OUT
os.environ["PAPER_GEN"] = OUT

In [ ]:
"""Quantify how much of each gene list survives a change of RNG stream.

The reference run (`data/reference_run_gene_lists.csv`) was produced by the same
pipeline, same seeds and same hyperparameters, executed on CUDA. The current run
executed on CPU. Weight initialisation is identical -- modules are built on CPU
and then moved -- so the divergence comes from dropout masks, which are drawn
from whichever device's generator is active. PLS-DA is unaffected because it is
deterministic; the CVAE is not.

Writes data/reproducibility.csv in the same key/value shape as
summary_scalars.csv, so make_macros.py picks it up unchanged.
"""
import os
import numpy as np
import pandas as pd
from pathlib import Path

if "PAPER_DATA" in os.environ:
    DATA = Path(os.environ["PAPER_DATA"])
else:
    ROOT = Path(__file__).resolve().parent.parent
    DATA = ROOT / "data"

# kaggle/build_notebook.py replaces this line with the literal contents of
# data/reference_run_gene_lists.csv, so the notebook carries the reference run
# with it and needs no extra dataset. Empty here: the local build reads the CSV.
# V6: the reference is v5's gene lists (the published run)
REFERENCE_ROWS = [
    ('CVAE_IG', 'KLRC1'),
    ('CVAE_IG', 'C2orf74'),
    ('CVAE_IG', 'SLC25A23'),
    ('CVAE_IG', 'OSCAR'),
    ('CVAE_IG', 'HLA-DQA2'),
    ('CVAE_IG', 'RCVRN'),
    ('CVAE_IG', 'GTF2H2C'),
    ('CVAE_IG', 'KYAT3'),
    ('CVAE_IG', 'CLEC7A'),
    ('CVAE_IG', 'SLX4'),
    ('CVAE_IG', 'HLA-DMA'),
    ('CVAE_IG', 'AHRR'),
    ('PLS-DA_VIP', 'CREBL2'),
    ('PLS-DA_VIP', 'SREK1'),
    ('PLS-DA_VIP', 'LPIN1'),
    ('PLS-DA_VIP', 'SAPCD2'),
    ('PLS-DA_VIP', 'LY6S-AS1'),
    ('PLS-DA_VIP', 'SLX4'),
    ('PLS-DA_VIP', 'FOXN3'),
    ('PLS-DA_VIP', 'ZHX1'),
    ('PLS-DA_VIP', 'DDX49'),
    ('PLS-DA_VIP', 'PPDPF'),
    ('PLS-DA_VIP', 'CHMP4C'),
    ('PLS-DA_VIP', 'TM7SF2'),
    ('PLS-DA_VIP', 'HLA-DQA2'),
    ('PLS-DA_VIP', 'PEX16'),
    ('PLS-DA_VIP', 'MFHAS1'),
    ('PLS-DA_VIP', 'PFKFB2'),
    ('PLS-DA_VIP', 'C2orf74'),
    ('PLS-DA_VIP', 'CINP'),
    ('PLS-DA_VIP', 'ZNF836'),
    ('PLS-DA_VIP', 'PDS5B'),
    ('PLS-DA_VIP', 'NEBL'),
    ('PLS-DA_VIP', 'AP1S2'),
    ('PLS-DA_VIP', 'ARL5A'),
    ('PLS-DA_VIP', 'ISG20L2'),
    ('PLS-DA_VIP', 'SESN1'),
    ('PLS-DA_VIP', 'ZMAT3'),
    ('PLS-DA_VIP', 'PLPPR1'),
    ('PLS-DA_VIP', 'MYOCD'),
    ('PLS-DA_VIP', 'DPH2'),
    ('PLS-DA_VIP', 'AGPAT1'),
    ('PLS-DA_VIP', 'NDUFB2-AS1'),
    ('PLS-DA_VIP', 'NR1D2'),
    ('PLS-DA_VIP', 'AHRR'),
    ('PLS-DA_VIP', 'TNR'),
    ('PLS-DA_VIP', 'GPD1'),
    ('PLS-DA_VIP', 'TMEM222'),
    ('PLS-DA_VIP', 'CNPY2'),
    ('PLS-DA_VIP', 'NEK11'),
    ('PLS-DA_VIP', 'RASAL1'),
    ('PLS-DA_VIP', 'SLC36A4'),
    ('PLS-DA_VIP', 'NKIRAS2'),
    ('PLS-DA_VIP', 'TMEM243'),
    ('PLS-DA_VIP', 'CREBRF'),
    ('PLS-DA_VIP', 'IRAK1'),
    ('PLS-DA_VIP', 'PPP4R3A'),
]

ref = (pd.DataFrame(REFERENCE_ROWS, columns=["method", "gene"]) if REFERENCE_ROWS
       else pd.read_csv(DATA / "reference_run_gene_lists.csv"))
rows = []


def put(k, v):
    rows.append({"key": k, "value": v})


for method, sig_file, tag in [("CVAE_IG", "CVAE_gene_significance.csv", "cvae"),
                              ("PLS-DA_VIP", "PLSDA_gene_significance.csv", "plsda")]:
    cur = pd.read_csv(DATA / sig_file)
    ranked = cur.sort_values("mean_score", ascending=False).reset_index(drop=True)
    rank = {g: i + 1 for i, g in enumerate(ranked.gene)}

    ref_set = set(ref.loc[ref.method == method, "gene"])
    cur_set = set(cur.loc[cur.significant, "gene"])
    shared = ref_set & cur_set

    put(f"repro_{tag}_n_reference", len(ref_set))
    put(f"repro_{tag}_n_current", len(cur_set))
    put(f"repro_{tag}_n_shared", len(shared))
    put(f"repro_{tag}_jaccard", len(shared) / max(len(ref_set | cur_set), 1))
    put(f"repro_{tag}_frac_current_in_reference",
        len(shared) / max(len(cur_set), 1))

    # where the reference run's genes land in this run's ranking
    ranks = np.array([rank[g] for g in ref_set if g in rank])
    if len(ranks):
        put(f"repro_{tag}_reference_median_rank_here", float(np.median(ranks)))
        put(f"repro_{tag}_reference_max_rank_here", int(ranks.max()))
        put(f"repro_{tag}_reference_frac_in_top_fifty",
            float((ranks <= 50).mean()))
        put(f"repro_{tag}_reference_frac_in_top_hundred",
            float((ranks <= 100).mean()))

pd.DataFrame(rows).to_csv(DATA / "reproducibility.csv", index=False)
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
"""Turn data/summary_scalars.csv into LaTeX macros.

Every number quoted in main.tex comes from here, so a reported value cannot
drift from the analysis that produced it. Keys are mangled to letter-only
macro names (LaTeX command names cannot contain digits or underscores):
`acc_pls_mean` -> `\vAccPlsMean`.
"""
import math
import os
import pandas as pd
from pathlib import Path

# Paths are overridable so the Kaggle notebook can run this file unmodified
# against /kaggle/working -- the same arrangement figstyle.py uses, and the
# reason the notebook emits the very file main.tex reads rather than a copy.
if "PAPER_DATA" in os.environ:
    DATA = Path(os.environ["PAPER_DATA"])
    GEN = Path(os.environ.get("PAPER_GEN", DATA / "generated"))
else:
    ROOT = Path(__file__).resolve().parent.parent
    DATA = ROOT / "data"
    GEN = ROOT / "generated"
GEN.mkdir(parents=True, exist_ok=True)

DIGITS = {"0": "Zero", "1": "One", "2": "Two", "3": "Three", "4": "Four",
          "5": "Five", "6": "Six", "7": "Seven", "8": "Eight", "9": "Nine"}


def macro_name(key: str) -> str:
    parts = [p for p in key.replace("-", "_").split("_") if p]
    name = "".join(p[:1].upper() + p[1:] for p in parts)
    name = "".join(DIGITS.get(c, c) for c in name)
    return "v" + "".join(ch for ch in name if ch.isalpha())


def sci(v: float, sig: int = 1) -> str:
    """Render as $a\\times 10^{b}$ with `sig` decimals on the mantissa."""
    if v == 0:
        return "0"
    e = int(math.floor(math.log10(abs(v))))
    m = v / (10 ** e)
    if round(m, sig) == 1 and sig == 0:
        return rf"10^{{{e}}}"
    return rf"{m:.{sig}f}\times 10^{{{e}}}"


def fmt(key: str, v) -> str:
    """Format by what the quantity is, not by its magnitude alone."""
    if isinstance(v, str):
        try:
            v = float(v)
        except ValueError:
            return v.replace("|", ", ").replace("_", r"\_")
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "--"

    k = key.lower()
    is_p = k.endswith("_p") or "_p_" in k or k.endswith("pvalue") or "_p_value" in k

    # counts stay integers
    if k.startswith("n_") or k.endswith("_n") or k.startswith("recon_n") or \
       k.startswith("unc_n") or k in ("n_genes", "n_subjects"):
        return f"{int(round(v)):,}".replace(",", r"{,}")

    if is_p:
        if v < 1e-4:
            return sci(v)
        return f"{v:.4f}".rstrip("0").rstrip(".")

    if abs(v) >= 1000:
        return f"{v:,.1f}".replace(",", r"{,}")
    if abs(v) >= 100:
        return f"{v:.1f}"
    if abs(v) >= 10:
        return f"{v:.2f}"
    if abs(v) < 1e-3 and v != 0:
        return sci(v, 2)
    return f"{v:.3f}"


def main():
    sources = ["summary_scalars.csv", "reproducibility.csv", "enrichment_counts.csv"]
    frames, used = [], []
    for s in sources:
        p = DATA / s
        if not p.exists():
            continue
        d = pd.read_csv(p)
        if s == "enrichment_counts.csv":         # reshape wide -> key/value
            d = pd.DataFrame([{"key": f"enr_{r['set']}_{c}", "value": r[c]}
                              for _, r in d.iterrows() for c in ("n_genes", "n_terms")])
        frames.append(d[["key", "value"]])
        used.append(s)
    df = pd.concat(frames, ignore_index=True)

    lines = ["% Generated by scripts/make_macros.py -- do not edit by hand.",
             "% Sources: " + ", ".join(f"data/{s}" for s in used), ""]
    seen = {}
    for _, r in df.iterrows():
        name = macro_name(str(r["key"]))
        if name in seen:                     # name collision after mangling
            raise SystemExit(f"macro collision: {r['key']} and {seen[name]} -> {name}")
        seen[name] = r["key"]
        lines.append(rf"\newcommand{{\{name}}}{{{fmt(str(r['key']), r['value'])}}}")

    (GEN / "values.tex").write_text("\n".join(lines) + "\n")
    print(f"wrote {GEN / 'values.tex'} with {len(seen)} macros")



main()

# show the macro file exactly as main.tex will read it
print(open(f"{OUT}/values.tex").read())

In [ ]:
"""Check this run's scalars against the ones the manuscript was written from.

A notebook that merely *produces* numbers is not evidence for a paper; a
notebook that produces them and then shows they are the published ones is. This
module reads `summary_scalars.csv` from the current run and compares it, key by
key, against a baseline snapshot of the run behind the manuscript.

`kaggle/build_notebook.py` replaces `BASELINE_ROWS` with the literal contents of
`data/summary_scalars.csv` when it inlines this file, so the notebook carries
the published values with it. Run locally with no injection, the baseline is
read from `data/` -- which is the same file as the current run, so the check is
degenerate but still exercises the comparator.

Three outcomes per key:
  identical  -- bit-for-bit equal (everything PLS-DA touches should land here)
  close      -- differs by less than RELATIVE_TOL, i.e. below reporting precision
  drifted    -- differs by more, and belongs in the discussion of the run
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd

if "PAPER_DATA" in os.environ:
    DATA = Path(os.environ["PAPER_DATA"])
else:
    DATA = Path(__file__).resolve().parent.parent / "data"

# Replaced at notebook-build time with [(key, value), ...] from the published
# data/summary_scalars.csv. Empty here so the local run falls back to the file.
BASELINE_ROWS = [
    ('acc_pls_mean', '0.8127450980392158'),
    ('acc_pls_sd', '0.04146051716390247'),
    ('auc_pls_mean', '0.8653571428571428'),
    ('auc_pls_sd', '0.04246531408228852'),
    ('ap_pls_mean', '0.7479193961906616'),
    ('ap_pls_sd', '0.09085595636501219'),
    ('acc_cvae_lat_true_mean', '0.7921568627450982'),
    ('acc_cvae_lat_true_sd', '0.045163052702936135'),
    ('auc_cvae_lat_true_mean', '0.8421428571428571'),
    ('auc_cvae_lat_true_sd', '0.05413984160467771'),
    ('ap_cvae_lat_true_mean', '0.7246941818587934'),
    ('ap_cvae_lat_true_sd', '0.09080731620121756'),
    ('acc_cvae_lat_neutral_mean', '0.7911764705882355'),
    ('acc_cvae_lat_neutral_sd', '0.04420070154921393'),
    ('auc_cvae_lat_neutral_mean', '0.8407142857142859'),
    ('auc_cvae_lat_neutral_sd', '0.05480748212306643'),
    ('ap_cvae_lat_neutral_mean', '0.7228082220613572'),
    ('ap_cvae_lat_neutral_sd', '0.09135122796426545'),
    ('acc_cvae_head_true_mean', '0.7911764705882354'),
    ('acc_cvae_head_true_sd', '0.051789639677834155'),
    ('auc_cvae_head_true_mean', '0.836875'),
    ('auc_cvae_head_true_sd', '0.05131617246958345'),
    ('acc_cvae_head_neutral_mean', '0.788235294117647'),
    ('acc_cvae_head_neutral_sd', '0.05152523022210089'),
    ('auc_cvae_head_neutral_mean', '0.8348214285714286'),
    ('auc_cvae_head_neutral_sd', '0.05142961569532949'),
    ('sil_pls_mean', '0.13979629158690474'),
    ('sil_pls_sd', '0.009456170188073598'),
    ('ari_pls_mean', '0.08205739685848476'),
    ('ari_pls_sd', '0.03556328469017687'),
    ('db_pls_mean', '1.4164704817029166'),
    ('db_pls_sd', '0.059467219084365606'),
    ('ch_pls_mean', '89.05322459376475'),
    ('ch_pls_sd', '6.443290846486978'),
    ('iir_pls_mean', '0.9229276054581058'),
    ('iir_pls_sd', '0.030914281713978742'),
    ('sil_cvae_true_mean', '0.41243404'),
    ('sil_cvae_true_sd', '0.14658120274543762'),
    ('ari_cvae_true_mean', '0.587146071928542'),
    ('ari_cvae_true_sd', '0.29286812897053904'),
    ('db_cvae_true_mean', '0.6486732695327644'),
    ('db_cvae_true_sd', '0.08370953355075654'),
    ('ch_cvae_true_mean', '344.69952'),
    ('ch_cvae_true_sd', '74.31301879882812'),
    ('iir_cvae_true_mean', '2.430014'),
    ('iir_cvae_true_sd', '0.6053985953330994'),
    ('sil_cvae_neutral_mean', '0.41145253'),
    ('sil_cvae_neutral_sd', '0.1461723893880844'),
    ('ari_cvae_neutral_mean', '0.5853630545758962'),
    ('ari_cvae_neutral_sd', '0.29155622369734585'),
    ('db_cvae_neutral_mean', '0.650215481544377'),
    ('db_cvae_neutral_sd', '0.08306436680615459'),
    ('ch_cvae_neutral_mean', '343.62787'),
    ('ch_cvae_neutral_sd', '73.92459106445312'),
    ('iir_cvae_neutral_mean', '2.4232333'),
    ('iir_cvae_neutral_sd', '0.6020241975784302'),
    ('mse_pls_mean_mean', '0.5410778384320537'),
    ('mse_pls_mean_sd', '0.04400760593456173'),
    ('mse_cvae_true_mean_mean', '0.84534216'),
    ('mse_cvae_true_mean_sd', '0.11572782695293427'),
    ('mse_cvae_neutral_mean_mean', '0.84464025'),
    ('mse_cvae_neutral_mean_sd', '0.11647577583789825'),
    ('acc_cvae_true_vs_pls_t', '-2.1462472202175036'),
    ('acc_cvae_true_vs_pls_p', '0.044980656513998594'),
    ('acc_cvae_true_vs_pls_cohen_dz', '-0.479915468081443'),
    ('auc_cvae_true_vs_pls_t', '-2.619372274250288'),
    ('auc_cvae_true_vs_pls_p', '0.016872351518284454'),
    ('auc_cvae_true_vs_pls_cohen_dz', '-0.5857094463454089'),
    ('acc_cvae_neutral_vs_pls_t', '-2.238015772274602'),
    ('acc_cvae_neutral_vs_pls_p', '0.037392500013527924'),
    ('acc_cvae_neutral_vs_pls_cohen_dz', '-0.5004355401406588'),
    ('auc_cvae_neutral_vs_pls_t', '-2.747992915540682'),
    ('auc_cvae_neutral_vs_pls_p', '0.012790497524458303'),
    ('auc_cvae_neutral_vs_pls_cohen_dz', '-0.6144698960683586'),
    ('acc_cvae_true_vs_neutral_t', '1.0'),
    ('acc_cvae_true_vs_neutral_p', '0.32987680092112504'),
    ('acc_cvae_true_vs_neutral_cohen_dz', '0.22360679769897895'),
    ('auc_cvae_true_vs_neutral_t', '3.469813245751993'),
    ('auc_cvae_true_vs_neutral_p', '0.0025651073253487268'),
    ('auc_cvae_true_vs_neutral_cohen_dz', '0.7758738282516776'),
    ('recon_cvae_true_vs_pls_wilcoxon_p', '1.8214101217267417e-168'),
    ('recon_cvae_true_vs_pls_ttest_p', '1.3930689868231686e-197'),
    ('recon_cvae_true_vs_pls_cohen_dz', '1.190206765557961'),
    ('recon_cvae_true_vs_pls_rank_biserial', '1.0'),
    ('recon_cvae_neutral_vs_pls_wilcoxon_p', '1.8214061814260869e-168'),
    ('recon_cvae_neutral_vs_pls_ttest_p', '8.07013513045763e-198'),
    ('recon_cvae_neutral_vs_pls_cohen_dz', '1.1912934361623668'),
    ('recon_cvae_neutral_vs_pls_rank_biserial', '1.0'),
    ('recon_n_oof_samples', '1020'),
    ('recon_mse_pls_Control_mean', '0.5449228709657026'),
    ('recon_mse_pls_Control_sd', '0.3019626730461255'),
    ('recon_mse_cvae_true_Control_mean', '0.8391556'),
    ('recon_mse_cvae_true_Control_sd', '0.4496098756790161'),
    ('recon_n_Control', '700'),
    ('recon_mse_pls_AD_mean', '0.5326668297646966'),
    ('recon_mse_pls_AD_sd', '0.22663110752778376'),
    ('recon_mse_cvae_true_AD_mean', '0.8588751'),
    ('recon_mse_cvae_true_AD_sd', '0.4057733118534088'),
    ('recon_n_AD', '320'),
    ('recon_group_pls_mannwhitney_p', '0.16569471725747542'),
    ('recon_group_cvaetrue_mannwhitney_p', '0.042911771224646955'),
    ('recon_group_cvaeneutral_mannwhitney_p', '0.03191187885671462'),
    ('cm_pls_sensitivity', '0.615625'),
    ('cm_pls_specificity', '0.9028571428571428'),
    ('cm_pls_precision', '0.7433962264150943'),
    ('cm_pls_balanced_accuracy', '0.7592410714285713'),
    ('cm_cvaetrue_sensitivity', '0.621875'),
    ('cm_cvaetrue_specificity', '0.87'),
    ('cm_cvaetrue_precision', '0.6862068965517242'),
    ('cm_cvaetrue_balanced_accuracy', '0.7459374999999999'),
    ('cm_cvaeneutral_sensitivity', '0.621875'),
    ('cm_cvaeneutral_specificity', '0.8685714285714285'),
    ('cm_cvaeneutral_precision', '0.6838487972508591'),
    ('cm_cvaeneutral_balanced_accuracy', '0.7452232142857143'),
    ('prob_pls_AD_mean', '0.6182651090051245'),
    ('prob_pls_control_mean', '0.1053383207914857'),
    ('prob_pls_gap', '0.5129267882136388'),
    ('prob_pls_mannwhitney_p', '1.7215701081463126e-77'),
    ('prob_pls_frac_ambiguous', '0.024509803921568627'),
    ('prob_cvaelat_AD_mean', '0.5964692321938596'),
    ('prob_cvaelat_control_mean', '0.1652155698305138'),
    ('prob_cvaelat_gap', '0.43125366236334584'),
    ('prob_cvaelat_mannwhitney_p', '1.2500645200650355e-63'),
    ('prob_cvaelat_frac_ambiguous', '0.06274509803921569'),
    ('prob_cvaehead_AD_mean', '0.6033827103230578'),
    ('prob_cvaehead_control_mean', '0.205003194985667'),
    ('prob_cvaehead_gap', '0.3983795153373908'),
    ('prob_cvaehead_mannwhitney_p', '9.097827387600113e-59'),
    ('prob_cvaehead_frac_ambiguous', '0.1323529411764706'),
    ('kl_true_AD_mean', '3.775103616900742'),
    ('kl_true_AD_median', '2.6325381994247437'),
    ('kl_true_control_mean', '5.5606403465781895'),
    ('kl_true_control_median', '3.0429261922836304'),
    ('kl_true_mannwhitney_p', '0.011903567544382706'),
    ('kl_true_pct_increase_AD', '-32.110271810263725'),
    ('meanvar_true_AD_mean', '0.8759011429734528'),
    ('meanvar_true_AD_median', '0.8585444092750549'),
    ('meanvar_true_control_mean', '0.848288049357278'),
    ('meanvar_true_control_median', '0.8353591859340668'),
    ('meanvar_true_mannwhitney_p', '1.0238000622109085e-05'),
    ('meanvar_true_pct_increase_AD', '3.2551553257288504'),
    ('kl_neutral_AD_mean', '3.7665718984790146'),
    ('kl_neutral_AD_median', '2.6133941411972046'),
    ('kl_neutral_control_mean', '5.5508226389118605'),
    ('kl_neutral_control_median', '3.0332001447677612'),
    ('kl_neutral_mannwhitney_p', '0.012004484726015029'),
    ('kl_neutral_pct_increase_AD', '-32.143897517550954'),
    ('meanvar_neutral_AD_mean', '0.8758510123938322'),
    ('meanvar_neutral_AD_median', '0.8585153818130493'),
    ('meanvar_neutral_control_mean', '0.8483885594350951'),
    ('meanvar_neutral_control_median', '0.8355096280574799'),
    ('meanvar_neutral_mannwhitney_p', '1.1282479035118865e-05'),
    ('meanvar_neutral_pct_increase_AD', '3.2370135892713146'),
    ('unc_n_AD', '320'),
    ('unc_n_control', '700'),
    ('n_sig_plsda', '45'),
    ('n_sig_cvae', '12'),
    ('n_overlap', '4'),
    ('jaccard', '0.07547169811320754'),
    ('consensus_genes', 'AHRR|C2orf74|HLA-DQA2|SLX4'),
    ('spearman_rho_vip_ig', '0.10060672494625085'),
    ('spearman_p_vip_ig', '2.1097418023635235e-49'),
    ('n_total_discovered', '53'),
    ('n_plsda_only', '41'),
    ('n_cvae_only', '8'),
    ('n_not_significant', '21409'),
    ('mean_stability_plsda', '0.658888888888889'),
    ('mean_stability_cvae', '0.5125000000000001'),
    ('n_genes', '21462'),
    ('n_subjects', '253'),
    ('mean_epochs_run', '30.55'),
]

RELATIVE_TOL = 1e-3      # the manuscript quotes 3-4 significant figures

# The numbers a reader would notice first if they moved.
HEADLINE = ["n_subjects", "n_genes", "acc_pls_mean", "auc_pls_mean",
            "acc_cvae_lat_true_mean", "auc_cvae_lat_true_mean",
            "sil_cvae_true_mean", "sil_pls_mean",
            "mse_pls_mean_mean", "mse_cvae_true_mean_mean",
            "n_sig_plsda", "n_sig_cvae", "n_overlap", "jaccard",
            "consensus_genes", "spearman_rho_vip_ig"]


def _num(v):
    """Float if the value is numeric, else None -- strings compare exactly."""
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    return f


def compare(baseline: pd.DataFrame, current: pd.DataFrame) -> pd.DataFrame:
    m = baseline.merge(current, on="key", how="outer",
                       suffixes=("_published", "_thisrun"))
    def absent(v):
        return v is None or (not isinstance(v, str) and pd.isna(v))

    out = []
    for _, r in m.iterrows():
        b, c = r["value_published"], r["value_thisrun"]
        bn, cn = _num(b), _num(c)
        if absent(b):
            status, adiff, rdiff = "new key", np.nan, np.nan
        elif absent(c):
            status, adiff, rdiff = "missing", np.nan, np.nan
        elif bn is None or cn is None:
            status = "identical" if str(b) == str(c) else "drifted"
            adiff = rdiff = np.nan
        else:
            adiff = abs(cn - bn)
            rdiff = adiff / max(abs(bn), 1e-12)
            if cn == bn:
                status = "identical"
            elif rdiff <= RELATIVE_TOL or adiff < 1e-12:
                status = "close"
            else:
                status = "drifted"
        out.append({"key": r["key"], "published": b, "this_run": c,
                    "abs_diff": adiff, "rel_diff": rdiff, "status": status})
    return pd.DataFrame(out)


def main():
    current = pd.read_csv(DATA / "summary_scalars.csv")[["key", "value"]]
    baseline = (pd.DataFrame(BASELINE_ROWS, columns=["key", "value"])
                if BASELINE_ROWS
                else pd.read_csv(DATA / "summary_scalars.csv")[["key", "value"]])

    chk = compare(baseline, current)
    chk.to_csv(DATA / "reproduction_check.csv", index=False)

    counts = chk.status.value_counts().to_dict()
    n = len(chk)
    agree = counts.get("identical", 0) + counts.get("close", 0)
    print(f"comparison with v5: {agree}/{n} scalars unchanged from v5 "
          f"(rel. tol {RELATIVE_TOL:g})")
    for k in ("identical", "close", "drifted", "missing", "new key"):
        if counts.get(k):
            print(f"  {k:10} {counts[k]}")

    print("\nheadline values")
    head = chk[chk.key.isin(HEADLINE)].set_index("key").reindex(HEADLINE).dropna(how="all")
    print(head[["published", "this_run", "status"]].to_string())

    drifted = chk[chk.status == "drifted"].sort_values(
        "rel_diff", ascending=False, na_position="last")
    if len(drifted):
        print(f"\n{len(drifted)} drifted scalars, largest relative change first")
        print(drifted[["key", "published", "this_run", "rel_diff"]]
              .head(40).to_string(index=False))
    else:
        print("\nno drift: identical to v5")
    return chk



_ = main()

---

# Figures

Everything below is the manuscript's own figure code, pasted in unchanged from
`scripts/figstyle.py`, `scripts/fig_design.py` and `scripts/make_figures.py`.
It reads only the CSV and `.npy` files written by the cells above, so each
figure in the paper can be traced to the analysis in this notebook and to
nothing else. The files are also written to `/kaggle/working/figures` as both
PDF and 400-dpi PNG.

In [ ]:
# Point the shared style module at this notebook's output directory and
# ask it to render inline, then run it exactly as the paper build does.
import os
os.environ["PAPER_DATA"] = OUT
os.environ["PAPER_FIGS"] = f"{OUT}/figures"
os.environ["PAPER_INLINE"] = "1"

"""Shared visual language for every figure in the paper.

One module so that a panel letter, a tick, a bar gap or a significance bracket
looks the same wherever it appears -- the thing that separates a figure set from
a pile of plots. Everything here is layout and typography; the numbers come from
../data and are never touched.

Colour is the data-viz reference palette used unchanged: categorical slots 1-3
(blue / orange / aqua) for the three model arms, which is the documented
all-pairs colourblind-safe subset, and the documented blue-red diverging pair
with a neutral grey midpoint for up/down regulation. Identity never rests on
colour alone -- every series also carries a marker shape, a direct label or a
legend entry.
"""
import os
import warnings
from pathlib import Path

import matplotlib as mpl
# PAPER_INLINE is set when this same code runs inside the Kaggle notebook, where
# the figures must render in the output rather than only be written to disk.
INLINE = bool(os.environ.get("PAPER_INLINE"))
if not INLINE:
    mpl.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

# Paths are overridable so that the notebook can point them at /kaggle/working
# and run this file unmodified -- one source of truth for every figure.
if "PAPER_DATA" in os.environ:
    DATA = Path(os.environ["PAPER_DATA"])
    FIG = Path(os.environ.get("PAPER_FIGS", DATA / "figures"))
else:
    ROOT = Path(__file__).resolve().parent.parent
    DATA = ROOT / "data"
    FIG = ROOT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# Target printed width. The LaTeX text block is 6.27 in wide, so a figure drawn
# much wider than this is downscaled by \includegraphics and every font in it
# shrinks with the drawing -- the usual reason a figure set looks amateur on the
# page. Everything here is laid out at TW and lands at ~0.9 scale.
TW = 6.9

# ---- palette -------------------------------------------------------------
# A cool, low-chroma set in the register of a printed journal figure: navy,
# wine and violet rather than the primary blue / orange / green of a screen
# palette. The three model arms stay separable for the common forms of colour
# blindness -- they differ in hue *and* in lightness (navy 30%, wine 42%,
# violet 55%), so they also survive a greyscale print -- and identity never
# rests on colour alone, since every series carries a marker or a direct label.
PLS     = "#24486b"   # categorical slot 1 -- navy
CVAE_T  = "#8f3a4f"   # slot 2 -- wine
CVAE_N  = "#6f6396"   # slot 3 -- muted violet
AD      = "#8f3a4f"   # diverging warm pole -- wine
CTRL    = "#24486b"   # diverging cool pole -- navy
UP, DOWN = AD, CTRL
MID     = "#8a8a92"   # neutral midpoint ink, faintly cool
ACCENT  = "#2f7d8c"   # reserved for "found by both" -- petrol blue
INK     = "#17171c"
INK2    = "#54545e"
INK3    = "#8a8a92"
RULE    = "#c9c9d1"
GRID    = "#e9e9ef"
SURFACE = "#ffffff"

# Diverging ramp for probability surfaces: cool pole -> neutral grey -> warm
# pole. Deliberately light at both ends -- this ramp is only ever a background
# under data marks, and a saturated pole swallows them.
DIVERGING = LinearSegmentedColormap.from_list(
    "ctrl_ad", ["#9aabc0", "#bcc6d5", "#dadfe7", "#eeeef1",
                "#e8d3d9", "#d5aeb9", "#bf8b9a"])

# Sequential ramps for the confusion matrices, built from the two arm colours so
# the matrices sit inside the same muted palette as everything else instead of
# borrowing matplotlib's saturated "Blues"/"Oranges".
SEQ_PLS = LinearSegmentedColormap.from_list(
    "seq_pls", ["#ffffff", "#e7ecf2", "#c3d1e0", "#93aac6", "#5c7fa6", PLS, "#173252"])
SEQ_CVAE = LinearSegmentedColormap.from_list(
    "seq_cvae", ["#ffffff", "#f5e9ec", "#e6c9d1", "#d0a1ae", "#b06c81", CVAE_T, "#63242f"])

MODEL_COLOR = {"pls": PLS, "cvae_true": CVAE_T, "cvae_neutral": CVAE_N,
               "PLS-DA": PLS, "CVAE_true": CVAE_T, "CVAE_neutral": CVAE_N}
MODEL_LABEL = {"pls": "PLS-DA",
               "cvae_true": "CVAE (label-conditioned)",
               "cvae_neutral": "CVAE (label-agnostic)",
               "PLS-DA": "PLS-DA",
               "CVAE_true": "CVAE (label-conditioned)",
               "CVAE_neutral": "CVAE (label-agnostic)"}
GROUP_COLOR = {"AD": AD, "Control": CTRL}
GROUP_MARKER = {"AD": "^", "Control": "o"}

# database sources in the g:Profiler output
SRC_COLOR = {"GO:MF": "#24486b", "GO:BP": "#2f7d8c", "GO:CC": "#6f6396",
             "CORUM": "#8f3a4f", "KEGG": "#4b7ca8", "REAC": "#57516e",
             "WP": "#3f6d80", "MIRNA": "#a3798f", "TF": "#54545e", "HP": "#7a4a5e"}

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "Latin Modern Roman", "DejaVu Serif", "Times New Roman"],
    "mathtext.fontset": "stix",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.titlelocation": "left",
    "axes.titlepad": 7,
    "axes.labelsize": 8,
    "axes.labelpad": 3.5,
    "axes.labelcolor": INK,
    "axes.edgecolor": RULE,
    "axes.linewidth": 0.6,
    "axes.grid": False,
    "grid.color": GRID,
    "grid.linewidth": 0.5,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.color": INK2, "ytick.color": INK2,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
    "xtick.major.size": 2.6, "ytick.major.size": 2.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.pad": 2.5, "ytick.major.pad": 2.5,
    "legend.fontsize": 7,
    "legend.frameon": False,
    "legend.handlelength": 1.1,
    "legend.handletextpad": 0.5,
    "legend.columnspacing": 1.2,
    "legend.borderaxespad": 0.2,
    "lines.linewidth": 1.4,
    "lines.solid_capstyle": "round",
    "patch.linewidth": 0,
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "pdf.fonttype": 42,
})


# ---- primitives ----------------------------------------------------------
def save(fig, name):
    """Vector PDF for pdflatex, 400-dpi PNG for the pandoc/docx build."""
    for ext, kw in (("pdf", {}), ("png", {"dpi": 400})):
        fig.savefig(FIG / f"{name}.{ext}", bbox_inches="tight",
                    pad_inches=0.03, **kw)
    if INLINE:
        try:
            from IPython.display import display
            display(fig)
        except ImportError:      # running the notebook's cells outside a kernel
            pass
    plt.close(fig)
    print(f"  {FIG.name}/{name}.pdf + .png")


def panel(ax, letter, title=None, subtitle=None, dx=-0.13, dy=1.0):
    """Panel letter in sans-serif, hung outside the axes at a fixed offset so
    the letters line up down the figure regardless of tick-label width.

    Title and subtitle are both annotations rather than `set_title`, because a
    real title always sits at one fixed height and would land on top of the
    subtitle whenever both are present.
    """
    ax.annotate(letter, xy=(dx, dy), xycoords="axes fraction",
                xytext=(0, 2), textcoords="offset points",
                fontsize=9.5, fontweight="bold", family="DejaVu Sans",
                color=INK, ha="left", va="bottom", annotation_clip=False)
    if title:
        ax.annotate(title, xy=(0, 1.0), xycoords="axes fraction",
                    xytext=(0, 13 if subtitle else 3), textcoords="offset points",
                    fontsize=8.6, color=INK, ha="left", va="bottom",
                    annotation_clip=False)
    if subtitle:
        ax.annotate(subtitle, xy=(0, 1.0), xycoords="axes fraction",
                    xytext=(0, 3), textcoords="offset points",
                    fontsize=6.8, color=INK3, ha="left", va="bottom",
                    annotation_clip=False)


def ygrid(ax):
    ax.grid(axis="y", visible=True)
    ax.grid(axis="x", visible=False)


def xgrid(ax):
    ax.grid(axis="x", visible=True)
    ax.grid(axis="y", visible=False)


def despine(ax, left=True, bottom=True):
    ax.spines["left"].set_visible(left)
    ax.spines["bottom"].set_visible(bottom)


def load(name):
    return pd.read_csv(DATA / name)


def scal(df=None):
    """summary_scalars.csv -> dict, numeric where possible."""
    df = df if df is not None else load("summary_scalars.csv")
    out = {}
    for _, r in df.iterrows():
        try:
            out[r["key"]] = float(r["value"])
        except (TypeError, ValueError):
            out[r["key"]] = r["value"]
    return out


def stars(p):
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "n.s."


def pmath(p, prefix="p"):
    """`$p = ...$` with a real times-ten-to-the exponent, not `1e-05`."""
    if p >= 1e-3:
        return f"${prefix} = {p:.3f}$"
    if not p > 1e-300:          # V7: with thousands of pairs a p-value can underflow to 0
        return f"${prefix} < 10^{{-300}}$"
    e = int(np.floor(np.log10(p)))
    return f"${prefix} = {p/10**e:.1f}\\times 10^{{{e}}}$"


def italic_ticks(ax, names, fontsize=6.0, axis="y"):
    """Italic gene symbols as real text, so hyphens stay hyphens and digits do
    not pick up math-italic spacing (`HLA-DQA2`, not `HLA - DQA2`)."""
    ticks = np.arange(len(names))
    if axis == "y":
        ax.set_yticks(ticks)
        ax.set_yticklabels(names, fontsize=fontsize, fontstyle="italic")
    else:
        ax.set_xticks(ticks)
        ax.set_xticklabels(names, fontsize=fontsize, fontstyle="italic")


def bracket(ax, x0, x1, y, text, tick=None, fontsize=7.2, color=INK):
    """Significance bracket that never overlaps the marks it spans."""
    tick = tick if tick is not None else abs(y) * 0.02 + 1e-9
    ax.plot([x0, x0, x1, x1], [y - tick, y, y, y - tick],
            color=color, lw=0.7, clip_on=False, solid_capstyle="butt")
    ax.text((x0 + x1) / 2, y + tick * 0.5, text, ha="center", va="bottom",
            fontsize=fontsize, color=color, clip_on=False)


def boxstrip(ax, values, pos, color, width=0.44, jitter=0.055, seed=0, size=4):
    """A box over its own points: the distribution and its summary together."""
    rs = np.random.RandomState(int(round(seed)))
    ax.scatter(rs.normal(pos, jitter, len(values)), values, s=size, color=color,
               alpha=0.28, linewidth=0, zorder=1, rasterized=True)
    bp = ax.boxplot([values], positions=[pos], widths=width, showfliers=False,
                    patch_artist=True, zorder=3, manage_ticks=False)
    bp["boxes"][0].set(facecolor=SURFACE, edgecolor=color, linewidth=1.2, alpha=0.95)
    for part in ("whiskers", "caps", "medians"):
        for a in bp[part]:
            a.set(color=color, linewidth=1.2)
    return bp


def swatches(labels_colors, marker="s", size=5):
    return [Line2D([], [], marker=marker, ls="", color=c, markersize=size, label=l)
            for l, c in labels_colors]


def repel_labels(ax, xs, ys, labels, fontsize=5.4, color=None, italic=True,
                 clearance=1.6, leader_color=None):
    """Place a label for every point without letting any two labels touch.

    A dependency-free replacement for adjustText: each label is tried at a ring
    of candidate offsets, in decreasing order of preference, and the first
    offset whose estimated box overlaps neither an already-placed label nor its
    own point is taken. Everything is measured in typographic points, which is
    exactly the unit `annotate` takes for `xytext`, so no round trip through
    data coordinates is needed and the result is stable at any figure size.
    """
    color = INK if color is None else color
    leader_color = RULE if leader_color is None else leader_color
    fig = ax.figure
    ppd = fig.dpi / 72.0                     # points -> display pixels
    pts = ax.transData.transform(np.column_stack([xs, ys]))

    # candidate offsets: near ring first, then progressively further out
    cands = []
    for radius in (7, 11, 16, 22, 30, 40, 52, 66, 82, 100):   # V7: wider search
        for ang in range(0, 360, 30):
            a = np.deg2rad(ang)
            cands.append((radius * np.cos(a), radius * np.sin(a)))

    placed = []                              # display-space boxes already taken
    leg = ax.get_legend()                    # a label under the key is unreadable
    if leg is not None:
        bb = leg.get_window_extent()
        placed.append((bb.x0, bb.y0, bb.x1, bb.y1))
    order = np.argsort(-np.asarray(ys))      # most significant genes get first pick

    def overlaps(box):
        x0, y0, x1, y1 = box
        for p0, q0, p1, q1 in placed:
            if x0 < p1 and p0 < x1 and y0 < q1 and q0 < y1:
                return True
        return False

    axbb = ax.get_window_extent()            # V7: labels must stay inside the axes
    inside = lambda b: (b[0] >= axbb.x0 and b[2] <= axbb.x1 and
                        b[1] >= axbb.y0 and b[3] <= axbb.y1)
    forced = 0
    for i in order:
        text = labels[i]
        w = len(text) * fontsize * 0.56 * ppd + clearance * ppd
        h = fontsize * 1.25 * ppd + clearance * ppd
        px, py = pts[i]
        boxes = []
        for dx, dy in cands:
            cx = px + dx * ppd + (w / 2 - clearance) * (1 if dx >= 0 else -1)
            cy = py + dy * ppd
            boxes.append(((cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2), dx, dy))
        ok = [t for t in boxes if inside(t[0]) and not overlaps(t[0])]
        if not ok:                           # never drop a gene: least-bad inside spot
            forced += 1
            ok = [t for t in boxes if inside(t[0])] or boxes
        for box, dx, dy in ok[:1]:
            if True:
                placed.append(box)
                ax.annotate(text, (xs[i], ys[i]), xytext=(dx, dy),
                            textcoords="offset points", fontsize=fontsize,
                            color=color, fontstyle="italic" if italic else "normal",
                            ha="left" if dx >= 0 else "right", va="center",
                            zorder=5, annotation_clip=False,
                            arrowprops=dict(arrowstyle="-", color=leader_color,
                                            lw=0.35, shrinkA=0.5, shrinkB=1.5))
    if forced:
        print(f"  repel_labels: {forced} label(s) had no collision-free spot")


def kde2d(xy, grid_x, grid_y, bw_scale=1.0):
    """Gaussian KDE evaluated on a grid, without a scipy import at call sites."""
    from scipy.stats import gaussian_kde
    k = gaussian_kde(xy.T)
    k.set_bandwidth(k.factor * bw_scale)
    XX, YY = np.meshgrid(grid_x, grid_y)
    return k(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Patch

# the two objects every figure function draws from
S = scal()
runs = load("run_metrics.csv")
print(f"{len(S)} scalars, {len(runs)} runs")

In [ ]:
"""Figure 1 -- the study design, drawn rather than measured.

The only figure in the paper that carries no data: it exists so the reader can
see, before any result, that the two models are matched on the thing that
matters (an 8-dimensional bottleneck fitted on identical splits) and differ only
in how they get there.
"""
import numpy as np
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle


BOX_KW = dict(boxstyle="round,pad=0.30,rounding_size=0.09", linewidth=0.9)


def box(ax, x, y, w, h, text, face, edge, fontsize=7.2, color=INK, weight="normal"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, facecolor=face, edgecolor=edge, **BOX_KW))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center",
            fontsize=fontsize, color=color, fontweight=weight, linespacing=1.35)


def arrow(ax, xy0, xy1, color=INK2, lw=0.9, style="-|>", rad=0.0, ls="-"):
    ax.add_patch(FancyArrowPatch(xy0, xy1, arrowstyle=style, mutation_scale=8,
                                 color=color, lw=lw, linestyle=ls,
                                 shrinkA=1.5, shrinkB=1.5,
                                 connectionstyle=f"arc3,rad={rad}"))


def tint(hex_color, alpha):
    """Flat tint over the white surface -- keeps the PDF free of transparency."""
    r, g, b = (int(hex_color[i:i + 2], 16) / 255 for i in (1, 3, 5))
    return tuple(1 - alpha * (1 - c) for c in (r, g, b))


def fig_design():
    coh = load("cohort_summary.csv").iloc[0]
    fig = plt.figure(figsize=(13.6, 7.2))
    gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 2.9], hspace=0.10)

    # ---------------- (a) cohort and resampling protocol -----------------
    ax = fig.add_subplot(gs[0])
    ax.set_xlim(0, 100); ax.set_ylim(0, 23); ax.axis("off")
    panel(ax, "a", dx=-0.012, dy=1.02)
    ax.text(0, 21.6, "Cohort, preprocessing and resampling protocol",
            fontsize=8.8, color=INK, ha="left", va="bottom")

    box(ax, 0.5, 8, 15, 11,
        f"{int(coh.n_cohorts)} GEO cohorts\n{int(coh.n_subjects)} arrays\n{int(coh.n_donors)} donors, age $\\geq${int(coh.age_min)}",
        tint(MID, 0.10), RULE)
    box(ax, 19, 8, 16, 11,
        f"Probe $\\rightarrow$ symbol\n(NCBI Gene, all cohorts)\n{int(coh.n_genes):,} genes",
        tint(MID, 0.10), RULE)
    box(ax, 38.5, 8, 17, 11,
        f"Donor-grouped split\n60 / 20 / 20\ntrain / val / test",
        tint(MID, 0.10), RULE)
    arrow(ax, (15.5, 13.5), (19, 13.5)); arrow(ax, (35, 13.5), (38.5, 13.5))

    # the two resampling arms
    box(ax, 60, 14, 18, 8, f"$\\times\\,20$ real-label splits\nrank statistics",
        tint(PLS, 0.13), PLS)
    box(ax, 60, 3.5, 18, 8, "$\\times\\,10$ label-permuted splits\nempirical null",
        tint(MID, 0.16), INK3)
    arrow(ax, (55.5, 14.5), (60, 18), rad=0.15)
    arrow(ax, (55.5, 12.5), (60, 7.5), rad=-0.15)

    box(ax, 82, 8, 17, 11,
        "One-sided Mann–Whitney\n$\\rightarrow$ BH FDR $q<0.05$\n$+$ stability selection",
        tint(CVAE_T, 0.13), CVAE_T)
    arrow(ax, (78, 18), (82, 14.6), rad=-0.15)
    arrow(ax, (78, 7.5), (82, 12.4), rad=0.15)

    ax.text(27, 5.4, "z-scored with training-fold parameters only",
            fontsize=6.6, color=INK3, ha="center", style="italic")

    # ---------------- (b) the two matched architectures ------------------
    ax = fig.add_subplot(gs[1])
    ax.set_xlim(0, 100); ax.set_ylim(-4.0, 57); ax.axis("off")
    panel(ax, "b", dx=-0.012, dy=1.01)
    ax.text(0, 55.2, "Two representations of the same 8-dimensional bottleneck",
            fontsize=8.8, color=INK, ha="left", va="bottom")

    # ---- the two shared inputs, drawn once on the left -------------------
    box(ax, 0.5, 21, 11.5, 10,
        f"$\\mathbf{{X}}$\n$n\\times{int(coh.n_genes):,}$\nexpression",
        tint(MID, 0.10), RULE, fontsize=7.0)
    box(ax, 0.5, 35, 11.5, 7.5, "$\\mathbf{Y}$\none-hot labels",
        tint(CVAE_N, 0.12), RULE, fontsize=7.0)
    ax.text(6.25, 19.6, "same split for both lanes", fontsize=6.2, color=INK3,
            ha="center", va="top", style="italic")

    # ================= PLS-DA lane =======================================
    lane = Rectangle((14.0, 29.5), 84.5, 25.0, facecolor=tint(PLS, 0.045),
                     edgecolor="none", zorder=0)
    ax.add_patch(lane)
    ax.text(15.0, 53.4, "PLS-DA  ·  linear, supervised projection",
            fontsize=7.6, color=PLS, ha="left", va="top", fontweight="bold")

    box(ax, 15.5, 38.0, 14.5, 9.0,
        "NIPALS\n$\\mathbf{T}=\\mathbf{XW}$\n8 components", tint(PLS, 0.16), PLS)
    box(ax, 33.5, 38.0, 12.5, 9.0, "Scores\n$\\mathbf{T}\\in\\mathbb{R}^{n\\times 8}$",
        tint(PLS, 0.30), PLS, fontsize=7.3, weight="bold")
    box(ax, 50.5, 44.6, 15.5, 6.0, "Logistic regression", tint(PLS, 0.14), PLS,
        fontsize=6.9)
    box(ax, 50.5, 36.8, 15.5, 6.4, "Loadings $\\mathbf{P}$\n$\\mathbf{T}\\mathbf{P}^{\\top}$",
        tint(PLS, 0.14), PLS, fontsize=6.9)
    box(ax, 70.0, 44.6, 14.0, 6.0, "$\\hat{y}$  prediction", tint(MID, 0.13), RULE,
        fontsize=6.9)
    box(ax, 70.0, 36.8, 14.0, 6.4, "$\\hat{\\mathbf{X}}$  reconstruction",
        tint(MID, 0.13), RULE, fontsize=6.9)
    box(ax, 33.5, 30.2, 12.5, 5.4, "x-weights $\\mathbf{W}$", tint(PLS, 0.10), PLS,
        fontsize=6.7)
    box(ax, 50.5, 30.2, 15.5, 5.4, "VIP score per gene", tint(PLS, 0.20), PLS,
        fontsize=6.9)

    arrow(ax, (12.0, 28.0), (15.5, 41.0), color=PLS, rad=0.14)
    arrow(ax, (12.0, 38.5), (15.5, 43.5), color=CVAE_N, rad=-0.10)
    arrow(ax, (30.0, 42.5), (33.5, 42.5), color=PLS)
    arrow(ax, (46.0, 43.6), (50.5, 47.6), color=PLS, rad=0.10)
    arrow(ax, (46.0, 41.5), (50.5, 40.0), color=PLS, rad=-0.10)
    arrow(ax, (66.0, 47.6), (70.0, 47.6), color=PLS)
    arrow(ax, (66.0, 40.0), (70.0, 40.0), color=PLS)
    arrow(ax, (39.7, 38.0), (39.7, 35.6), color=PLS, ls=(0, (2.4, 1.6)))
    arrow(ax, (46.0, 32.9), (50.5, 32.9), color=PLS)

    # ================= beta-CVAE lane ====================================
    lane = Rectangle((14.0, 1.0), 84.5, 25.0, facecolor=tint(CVAE_T, 0.045),
                     edgecolor="none", zorder=0)
    ax.add_patch(lane)
    ax.text(15.0, 24.9, "$\\beta$-CVAE  ·  non-linear, semi-supervised generative model",
            fontsize=7.6, color=CVAE_T, ha="left", va="top", fontweight="bold")

    box(ax, 15.5, 11.6, 14.0, 9.0,
        "Encoder $q_\\phi(\\mathbf{z}\\mid\\mathbf{x},\\mathbf{c})$\n"
        "$256\\rightarrow$ BN, ReLU\ndropout 0.2", tint(CVAE_T, 0.16), CVAE_T,
        fontsize=6.8)
    box(ax, 32.5, 16.4, 9.5, 4.6, "$\\mu$", tint(CVAE_T, 0.16), CVAE_T, fontsize=7.2)
    box(ax, 32.5, 10.6, 9.5, 4.6, "$\\log\\sigma^{2}$", tint(CVAE_T, 0.16), CVAE_T,
        fontsize=7.2)
    box(ax, 45.5, 12.6, 13.5, 6.8,
        "$\\mathbf{z}=\\mu+\\sigma\\odot\\epsilon$\n$\\in\\mathbb{R}^{n\\times 8}$",
        tint(CVAE_T, 0.30), CVAE_T, fontsize=7.2, weight="bold")
    box(ax, 45.5, 5.4, 13.5, 4.6, "$\\epsilon\\sim\\mathcal{N}(0,\\mathbf{I})$",
        tint(MID, 0.10), INK3, fontsize=6.7)
    box(ax, 62.5, 17.4, 14.0, 6.2, "Classifier head\nFC-ReLU-FC", tint(CVAE_T, 0.14),
        CVAE_T, fontsize=6.7)
    box(ax, 62.5, 9.6, 14.0, 6.2, "Decoder $p_\\theta(\\mathbf{x}\\mid\\mathbf{z},\\mathbf{c})$",
        tint(CVAE_T, 0.14), CVAE_T, fontsize=6.7)
    box(ax, 80.5, 17.4, 14.5, 6.2, "$\\hat{y}$  prediction", tint(MID, 0.13), RULE,
        fontsize=6.9)
    box(ax, 80.5, 9.6, 14.5, 6.2, "$\\hat{\\mathbf{X}}$  reconstruction",
        tint(MID, 0.13), RULE, fontsize=6.9)
    box(ax, 80.5, 2.0, 14.5, 5.6, "Integrated Gradients\nper gene",
        tint(CVAE_T, 0.20), CVAE_T, fontsize=6.7)

    arrow(ax, (6.25, 21.0), (15.5, 17.5), color=CVAE_T, rad=-0.10)
    arrow(ax, (29.5, 17.5), (32.5, 18.7), color=CVAE_T)
    arrow(ax, (29.5, 14.5), (32.5, 12.9), color=CVAE_T)
    arrow(ax, (42.0, 18.7), (45.5, 17.2), color=CVAE_T)
    arrow(ax, (42.0, 12.9), (45.5, 14.6), color=CVAE_T)
    arrow(ax, (52.2, 10.0), (52.2, 12.6), color=INK3, ls=(0, (2.4, 1.6)))
    arrow(ax, (59.0, 17.4), (62.5, 20.0), color=CVAE_T, rad=0.10)
    arrow(ax, (59.0, 14.6), (62.5, 12.7), color=CVAE_T, rad=-0.10)
    arrow(ax, (76.5, 20.5), (80.5, 20.5), color=CVAE_T)
    arrow(ax, (76.5, 12.7), (80.5, 12.7), color=CVAE_T)
    arrow(ax, (87.7, 17.4), (87.7, 7.6), color=CVAE_T, ls=(0, (2.4, 1.6)))

    # the conditioning vector: the one input the two lanes do not share
    box(ax, 15.5, 3.4, 14.0, 5.4, "$\\mathbf{c}$  conditioning", tint(CVAE_N, 0.18),
        CVAE_N, fontsize=6.8)
    arrow(ax, (22.5, 8.8), (22.5, 11.6), color=CVAE_N, ls=(0, (2.4, 1.6)))
    # the conditioning route to the decoder runs beneath the epsilon box
    arrow(ax, (29.5, 4.4), (69.5, 4.4), color=CVAE_N, ls=(0, (2.4, 1.6)), style="-")
    arrow(ax, (69.5, 4.4), (69.5, 9.6), color=CVAE_N, ls=(0, (2.4, 1.6)))
    ax.text(31.0, 2.6, "true one-hot label  |  neutral $1/K$",
            fontsize=6.2, color=CVAE_N, ha="left", va="center")

    # the matched constraint, drawn once between the two bottleneck boxes
    ax.annotate("", xy=(35.0, 38.2), xytext=(48.5, 19.6),
                arrowprops=dict(arrowstyle="<->", color=INK3, lw=0.8,
                                linestyle=(0, (2.6, 1.8)), shrinkA=2, shrinkB=2,
                                connectionstyle="arc3,rad=0.16"))
    ax.text(15.5, 27.9, "matched: $d=8$, identical folds",
            fontsize=6.8, color=INK2, ha="left", va="center", style="italic")

    # One rule, stated once: grey is data entering or leaving a lane, violet is
    # label information, a lane-coloured box is an operation of that lane, and
    # the one box per lane with a heavier fill is the matched bottleneck. Every
    # box in the drawing follows it, which is what makes the key worth reading.
    key = [("data in / out", [(tint(MID, 0.11), RULE)]),
           ("label information", [(tint(CVAE_N, 0.16), CVAE_N)]),
           ("operation", [(tint(PLS, 0.16), PLS), (tint(CVAE_T, 0.16), CVAE_T)]),
           ("matched $d=8$ bottleneck", [(tint(PLS, 0.30), PLS),
                                         (tint(CVAE_T, 0.30), CVAE_T)])]
    x0 = 15.0
    for lab, sw in key:
        for j, (fc, ec) in enumerate(sw):
            ax.add_patch(Rectangle((x0 + j * 2.4, -0.9), 2.1, 1.6, facecolor=fc,
                                   edgecolor=ec, linewidth=0.7, clip_on=False))
        x0 += 2.4 * (len(sw) - 1) + 3.0
        ax.text(x0, -0.1, lab, fontsize=6.4, color=INK2, ha="left", va="center")
        x0 += len(lab) * 0.95 + 5.0
    ax.text(15.0, -3.0, "Lane colour marks which model an operation belongs to; "
            "a heavier fill marks the bottleneck the two models share.",
            fontsize=6.2, color=INK3, ha="left", va="center", style="italic")

    save(fig, "fig1_design")



fig_design()

In [ ]:
# =====================================================================
# Figure 2 -- classification
# =====================================================================
def fig_classification(runs, S):
    roc, pr = load("roc_curves_raw.csv"), load("pr_curves_raw.csv")
    cm = load("confusion_matrices.csv").set_index("model")
    base = np.linspace(0, 1, 101)
    arms = ("pls", "cvae_true", "cvae_neutral")

    fig = plt.figure(figsize=(TW, 8.4))
    gs = fig.add_gridspec(4, 2, hspace=0.62, wspace=0.34,
                          height_ratios=[1, 1, 0.92, 1.02])

    # ---- (a) ROC -----------------------------------------------------
    ax = fig.add_subplot(gs[0, 0])
    for m in arms:
        v = roc[roc.model == m][[f"tpr_{i}" for i in range(101)]].values
        mu, sd = v.mean(0), v.std(0)
        key = "auc_pls" if m == "pls" else "auc_cvae_lat_" + m.split("_", 1)[1]
        ax.fill_between(base, mu - sd, mu + sd, color=MODEL_COLOR[m], alpha=0.12, lw=0)
        ax.plot(base, mu, color=MODEL_COLOR[m], lw=1.5,
                label=f"{MODEL_LABEL[m]}  {S[key + '_mean']:.3f}")
    ax.plot([0, 1], [0, 1], color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.004)
    ax.set_xticks([0, .25, .5, .75, 1]); ax.set_yticks([0, .25, .5, .75, 1])
    ax.legend(loc="lower right", title="mean ROC-AUC", title_fontsize=6.6,
              alignment="left", frameon=True, framealpha=0.92,
              facecolor=SURFACE, edgecolor="none")
    panel(ax, "a", "Receiver operating characteristic", "mean $\\pm$ SD, 20 held-out splits")

    # ---- (b) precision-recall ---------------------------------------
    ax = fig.add_subplot(gs[0, 1])
    for m in arms:
        v = pr[pr.model == m][[f"prec_{i}" for i in range(101)]].values
        mu, sd = v.mean(0), v.std(0)
        ax.fill_between(base, mu - sd, mu + sd, color=MODEL_COLOR[m], alpha=0.12, lw=0)
        ax.plot(base, mu, color=MODEL_COLOR[m], lw=1.5, label=MODEL_LABEL[m])
    prev = S["n_AD"] / S["n_subjects"] if "n_AD" in S else 0.5
    ax.axhline(prev, color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
    ax.text(0.015, prev - 0.02, "AD prevalence", fontsize=6.2, color=INK3,
            ha="left", va="top")                 # V7: left end, below the line, clear of curves
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.004)
    ax.set_xticks([0, .25, .5, .75, 1]); ax.set_yticks([0, .25, .5, .75, 1])
    ax.legend(loc="lower left", bbox_to_anchor=(-0.012, -0.028),
              frameon=True, framealpha=0.92, facecolor=SURFACE, edgecolor="none")
    panel(ax, "b", "Precision-recall")

    # ---- (c) the same split, both models ----------------------------
    ax = fig.add_subplot(gs[1, 0])
    order = np.argsort(runs.auc_pls.values)
    a, b = runs.auc_pls.values[order], runs.auc_cvae_lat_true.values[order]
    x = np.arange(len(order))
    for i in x:
        ax.plot([i, i], [a[i], b[i]], color=RULE, lw=0.8, zorder=1,
                solid_capstyle="round")
    ax.scatter(x, a, s=17, color=PLS, zorder=3, label=MODEL_LABEL["pls"],
               edgecolor=SURFACE, linewidth=0.7)
    ax.scatter(x, b, s=17, color=CVAE_T, zorder=3, marker="s",
               label=MODEL_LABEL["cvae_true"], edgecolor=SURFACE, linewidth=0.7)
    ax.set_xlabel("Split, ordered by PLS-DA AUC"); ax.set_ylabel("Test ROC-AUC")
    ax.set_xticks([]); ygrid(ax)
    p = S["auc_cvae_true_vs_pls_p"]
    ax.text(0.98, 0.03, f"paired $t$-test  {pmath(p)} ({stars(p)})\n"
                        f"PLS-DA higher in {int((a > b).sum())} of {len(x)} splits",
            transform=ax.transAxes, fontsize=6.6, va="bottom", ha="right", color=INK2)
    _lo, _hi = ax.get_ylim()
    ax.set_ylim(_lo, _hi + 0.32 * (_hi - _lo))      # V7: headroom so the key covers no split
    ax.legend(loc="upper left", ncol=2)
    panel(ax, "c", "Split-by-split, paired")

    # ---- (d) headline metrics ---------------------------------------
    ax = fig.add_subplot(gs[1, 1])
    metrics = [("Accuracy", "acc"), ("ROC-AUC", "auc"), ("Avg. precision", "ap")]
    keys = {"pls": "pls", "cvae_true": "cvae_lat_true", "cvae_neutral": "cvae_lat_neutral"}
    short = {"pls": "PLS", "cvae_true": "cond.", "cvae_neutral": "agn."}
    w = 0.26
    # every bar is named on its own tick, so this panel needs no legend -- a
    # fourth copy of the arm key would only fight the panels beneath it
    for j, arm in enumerate(arms):
        xs = np.arange(len(metrics)) + (j - 1) * w
        vals = [S[f"{m}_{keys[arm]}_mean"] for _, m in metrics]
        errs = [S[f"{m}_{keys[arm]}_sd"] for _, m in metrics]
        ax.bar(xs, vals, w * 0.86, yerr=errs, capsize=1.8, color=MODEL_COLOR[arm],
               edgecolor=SURFACE, linewidth=1.0, error_kw=dict(lw=0.7, ecolor=INK2))
    ax.set_xticks([i + (j - 1) * w for i in range(len(metrics)) for j in range(3)])
    ax.set_xticklabels([short[a] for _ in metrics for a in arms], fontsize=5.8)
    ax.tick_params(axis="x", length=0, pad=1.5)
    for i, (name, _) in enumerate(metrics):
        ax.text(i, -0.135, name, transform=ax.get_xaxis_transform(), ha="center",
                va="top", fontsize=7, color=INK)
    ax.set_ylim(0.6, 1.0); ax.set_ylabel("Test-set score")
    ax.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0]); ygrid(ax)
    panel(ax, "d", "Mean $\\pm$ SD over 20 splits",
          "PLS-DA / CVAE label-conditioned / label-agnostic; axis starts at 0.6")

    # ---- (e,f) confusion matrices ------------------------------------
    for i, m in enumerate(("pls", "cvae_true")):
        ax = fig.add_subplot(gs[2, i])
        r = cm.loc[m]
        M = np.array([[r.tn, r.fp], [r.fn, r.tp]], float)
        Mn = M / M.sum(1, keepdims=True)
        ax.imshow(Mn, cmap=SEQ_PLS if m == "pls" else SEQ_CVAE, vmin=0, vmax=1)
        # the rate is the number being read, so it is set large in the figure's
        # own serif with lining figures; the raw count is a subordinate line
        # beneath it rather than a second number of equal weight
        for u in range(2):
            for v in range(2):
                fg = SURFACE if Mn[u, v] > 0.55 else INK
                sub_fg = SURFACE if Mn[u, v] > 0.55 else INK2
                ax.text(v, u - 0.03, f"{Mn[u, v]:.2f}", ha="center", va="center",
                        fontsize=11.5, color=fg, family="serif",
                        fontweight="medium")
                ax.text(v, u + 0.24, f"n = {int(M[u, v]):,}", ha="center", va="center",
                        fontsize=6.4, color=sub_fg, family="serif")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Control", "AD"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["Control", "AD"])
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        for sp in ax.spines.values():
            sp.set_visible(False)
        ax.tick_params(length=0)
        tag = m.replace("_", "")
        panel(ax, "ef"[i], MODEL_LABEL[m],
              f"sensitivity {S[f'cm_{tag}_sensitivity']:.3f}   "
              f"specificity {S[f'cm_{tag}_specificity']:.3f}")

    # ---- (g) out-of-fold posterior confidence ------------------------
    ax = fig.add_subplot(gs[3, :])
    op = load("oof_predictions.csv")
    grid = np.linspace(0, 1, 300)
    from scipy.stats import gaussian_kde
    for j, (col, name, ls) in enumerate((("p_pls", "PLS-DA", "-"),
                                         ("p_cvae_lat_true", "CVAE", (0, (4, 2))))):
        for g in ("Control", "AD"):
            v = op.loc[op.group == g, col].values
            d = gaussian_kde(v)(grid)
            ax.plot(grid, d, color=GROUP_COLOR[g], lw=1.5, ls=ls)
            if j == 0:
                ax.fill_between(grid, 0, d, color=GROUP_COLOR[g], alpha=0.10, lw=0)
    ax.axvline(0.5, color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
    ax.set_xlim(0, 1); ax.set_xlabel("Predicted $P(\\mathrm{AD})$ on held-out subjects")
    ax.set_ylabel("Density"); ygrid(ax)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.45)          # V7: headroom above the density peaks
    ax.legend(handles=[*swatches([("Control", CTRL), ("AD", AD)]),
                       Line2D([], [], color=INK2, ls="-", label="PLS-DA"),
                       Line2D([], [], color=INK2, ls=(0, (4, 2)), label="CVAE")],
              loc="upper right", ncol=2, bbox_to_anchor=(0.99, 1.0),
              framealpha=0.9, frameon=True, facecolor=SURFACE, edgecolor="none")
    ax.text(0.30, 0.92, "mean $P(\\mathrm{AD})$, AD $-$ control\n"
                        f"PLS-DA  {S['prob_pls_gap']:+.3f}\n"
                        f"CVAE     {S['prob_cvaelat_gap']:+.3f}",
            transform=ax.transAxes, fontsize=6.6, color=INK2, va="top")
    panel(ax, "g", "Confidence calibration",
          f"all {int(S['recon_n_oof_samples']):,} held-out subject-fold pairs",
          dx=-0.045)

    save(fig, "fig2_classification")



fig_classification(runs, S)

In [ ]:
# =====================================================================
# Figure 3 -- latent geometry
# =====================================================================
def fig_geometry():
    emb = load("latent_embeddings.csv")
    tags = ["PLS-DA", "CVAE_true", "CVAE_neutral"]
    projs = [(("pca1", "pca2"), "PC"), (("tsne1", "tsne2"), "t-SNE")]
    if "umap1" in emb.columns and emb.umap1.notna().any():
        projs.append((("umap1", "umap2"), "UMAP"))

    fig, axes = plt.subplots(len(projs), 3, figsize=(TW, TW / 3 * len(projs) + 0.55))
    axes = np.atleast_2d(axes)
    for row, (proj, name) in enumerate(projs):
        for col, tag in enumerate(tags):
            ax = axes[row, col]
            sub = emb[emb.model == tag]
            for g in ("Control", "AD"):
                s = sub[sub.group == g]
                ax.scatter(s[proj[0]], s[proj[1]], s=15, color=GROUP_COLOR[g],
                           marker=GROUP_MARKER[g], alpha=0.82, linewidth=0.5,
                           edgecolor=SURFACE, label=g)
            # axis labels would only repeat the row label; the coordinates of an
            # embedding carry no unit worth printing twice
            ax.set_xticks([]); ax.set_yticks([])
            despine(ax)
            if row == 0 and col == 0:
                ax.legend(loc="upper left", markerscale=1.15, frameon=True, framealpha=0.92,
                          facecolor=SURFACE, edgecolor="none")
            panel(ax, "abcdefghi"[row * 3 + col],
                  MODEL_LABEL[tag] if row == 0 else None, dx=-0.11)   # V7: letter clear of title
        axes[row, 0].annotate(f"{name}\nprojection", xy=(-0.085, 0.5),
                              xycoords="axes fraction", rotation=90,
                              ha="center", va="center", fontsize=8.4,
                              color=INK2, annotation_clip=False)
    fig.tight_layout(w_pad=1.0, h_pad=1.8)
    save(fig, "fig3_latent_geometry")



fig_geometry()

In [ ]:
# =====================================================================
# Figure 4 -- what the geometry is worth, and what conditioning buys
# =====================================================================
def fig_separation(S):
    specs = [("sil", "Silhouette", "higher = better"),
             ("ari", "Adjusted Rand", "higher = better"),
             ("iir", "Inter / intra distance", "higher = better"),
             ("ch", "Calinski-Harabasz", "higher = better"),
             ("db", "Davies-Bouldin", "lower = better")]
    arms = [("pls", "PLS-DA"), ("cvae_true", "CVAE\ncond."), ("cvae_neutral", "CVAE\nagn.")]

    fig, axes = plt.subplots(2, 3, figsize=(TW, 5.0))
    axes = axes.ravel()
    axes[-1].axis("off")
    for i, (k, name, note) in enumerate(specs):
        ax = axes[i]
        for j, (sfx, _) in enumerate(arms):
            ax.bar(j, S[f"{k}_{sfx}_mean"], 0.60, yerr=S[f"{k}_{sfx}_sd"], capsize=2.0,
                   color=MODEL_COLOR[sfx], edgecolor=SURFACE, linewidth=1.0,
                   error_kw=dict(lw=0.7, ecolor=INK2))
        ax.set_xticks(range(3))
        ax.set_xticklabels([n for _, n in arms], fontsize=6.8)
        ygrid(ax)   # the panel title already names the metric
        panel(ax, "abcde"[i], name, note, dx=-0.22)
    axes[-1].text(0.02, 0.62, "Mean $\\pm$ SD over the 20 splits.\n\n"
                  "The label-agnostic arm re-encodes\nthe same fitted model with a\n"
                  "uniform conditioning vector, so any\ngap between the second and third\n"
                  "bar of a panel would be label leak;\ncompare them panel by panel.",
                  transform=axes[-1].transAxes, fontsize=7.2, color=INK2,
                  va="top", style="italic", linespacing=1.5)
    fig.tight_layout(w_pad=2.6, h_pad=3.0)
    save(fig, "fig4_latent_separation")



fig_separation(S)

In [ ]:
# =====================================================================
# Figure 5 -- posterior probability landscapes over the latent spaces
# =====================================================================
def fig_landscape():
    emb = load("latent_embeddings.csv")
    tags = ["PLS-DA", "CVAE_true"]
    projs = [(("pca1", "pca2"), "PCA"), (("tsne1", "tsne2"), "t-SNE")]
    if "umap1" in emb.columns and emb.umap1.notna().any():
        projs.append((("umap1", "umap2"), "UMAP"))

    fig, axes = plt.subplots(2, len(projs), figsize=(TW, TW / len(projs) * 2 + 0.7))
    axes = np.atleast_2d(axes)
    im = None
    for row, tag in enumerate(tags):
        for col, (proj, name) in enumerate(projs):
            ax = axes[row, col]
            sub = emb[emb.model == tag]
            XY = sub[list(proj)].values
            pad = 0.09 * (XY.max(0) - XY.min(0))
            gx = np.linspace(XY[:, 0].min() - pad[0], XY[:, 0].max() + pad[0], 220)
            gy = np.linspace(XY[:, 1].min() - pad[1], XY[:, 1].max() + pad[1], 220)
            # class-conditional KDEs -> posterior P(AD | position) at equal priors
            dens = {g: kde2d(sub.loc[sub.group == g, list(proj)].values, gx, gy, 1.15)
                    for g in ("AD", "Control")}
            post = dens["AD"] / (dens["AD"] + dens["Control"] + 1e-300)
            im = ax.pcolormesh(gx, gy, post, cmap=DIVERGING, vmin=0, vmax=1,
                               shading="gouraud", rasterized=True)
            ax.contour(gx, gy, post, levels=[0.5], colors=[INK2], linewidths=1.0,
                       linestyles=[(0, (3.5, 2.0))])
            # The surface already carries class by colour, so the marks must not:
            # a blue dot on a blue field is invisible. Shape plus fill/no-fill in
            # ink reads on every part of the ramp.
            mark = {"Control": dict(marker="o", facecolor=SURFACE, edgecolor=INK),
                    "AD": dict(marker="^", facecolor=INK, edgecolor=SURFACE)}
            for g in ("Control", "AD"):
                s = sub[sub.group == g]
                ax.scatter(s[proj[0]], s[proj[1]], s=13, linewidth=0.6,
                           label=g, zorder=3, **mark[g])
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_xlim(gx[0], gx[-1]); ax.set_ylim(gy[0], gy[-1])
            for sp in ax.spines.values():
                sp.set_visible(False)
            panel(ax, "abcdef"[row * len(projs) + col],
                  name if row == 0 else None, dx=-0.11)
            if col == 0:
                ax.annotate(MODEL_LABEL[tag], xy=(-0.06, 0.5),
                            xycoords="axes fraction", rotation=90, ha="center",
                            va="center", fontsize=7.6, color=INK2,
                            annotation_clip=False)
            if row == 0 and col == 0:
                ax.legend(loc="upper left", markerscale=1.25, labelcolor=INK,
                          facecolor=SURFACE, framealpha=0.88, frameon=True,
                          edgecolor="none", borderpad=0.45)
    cax = fig.add_axes([0.24, -0.022, 0.44, 0.012])
    cb = fig.colorbar(im, cax=cax, orientation="horizontal")
    cb.set_label("Posterior $P(\\mathrm{AD}\\mid$ latent position$)$ from class-conditional KDE",
                 fontsize=7, color=INK2)
    cb.set_ticks([0, 0.25, 0.5, 0.75, 1]); cb.ax.tick_params(labelsize=6.5, length=2)
    cb.outline.set_visible(False)
    fig.text(0.72, -0.020, "dashed contour: $P=0.5$", fontsize=6.6, color=INK2,
             ha="left", style="italic")
    fig.tight_layout(w_pad=1.0, h_pad=1.4)
    save(fig, "fig5_probability_landscape")



fig_landscape()

In [ ]:
# =====================================================================
# Figure 6 -- reconstruction fidelity
# =====================================================================
def fig_reconstruction(S):
    sc = load("recon_scatter_representative.csv")
    oof = load("oof_reconstruction.csv")

    fig = plt.figure(figsize=(TW, 6.4))
    gs = fig.add_gridspec(2, 2, wspace=0.34, hspace=0.52)
    lim = (min(sc.observed.min(), sc.recon_pls.min(), sc.recon_cvae.min()),
           max(sc.observed.max(), sc.recon_pls.max(), sc.recon_cvae.max()))

    for i, (col, name, cmap, key) in enumerate(
            (("recon_pls", "PLS-DA", SEQ_PLS, "mse_pls_mean_mean"),
             ("recon_cvae", "CVAE (label-conditioned)", SEQ_CVAE, "mse_cvae_true_mean_mean"))):
        ax = fig.add_subplot(gs[0, i])
        ax.hexbin(sc.observed, sc[col], gridsize=56, bins="log", cmap=cmap,
                  mincnt=1, linewidths=0, rasterized=True)
        ax.plot(lim, lim, color=INK, lw=0.8, ls=(0, (3.5, 2.5)))
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel("Observed expression ($z$)")
        ax.set_ylabel("Reconstructed ($z$)" if i == 0 else "")
        ax.text(0.96, 0.06, f"mean MSE {S[key]:.3f}", transform=ax.transAxes,
                ha="right", fontsize=7.2, color=INK)
        panel(ax, "ab"[i], name, "identity line dashed" if i == 0 else None)

    # (c) paired per-subject error
    ax = fig.add_subplot(gs[1, 0])
    d = {"PLS-DA": oof.mse_pls.values, "CVAE": oof.mse_cvae_true.values}
    for j, (nm, color) in enumerate((("PLS-DA", PLS), ("CVAE", CVAE_T))):
        boxstrip(ax, d[nm], j, color, seed=j, size=3.5)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["PLS-DA", "CVAE"])
    ax.set_yscale("log"); ax.set_ylabel("Per-subject MSE (log)")
    ygrid(ax)
    p, r = S["recon_cvae_true_vs_pls_wilcoxon_p"], S["recon_cvae_true_vs_pls_rank_biserial"]
    top = max(v.max() for v in d.values())
    bracket(ax, 0, 1, top * 1.45, f"{stars(p)}  Wilcoxon {pmath(p)},  $r={r:.2f}$",
            tick=top * 0.30, fontsize=6.8)
    ax.set_ylim(top=top * 4.2)
    panel(ax, "c", "Every held-out subject-fold pair",
          f"$n = {int(S['recon_n_oof_samples']):,}$, paired")

    # (d) does either model reconstruct one diagnosis better?
    ax = fig.add_subplot(gs[1, 1])
    pos = 0
    ticks, labels = [], []
    for nm, col, color, tag in (("PLS-DA", "mse_pls", PLS, "pls"),
                                ("CVAE", "mse_cvae_true", CVAE_T, "cvaetrue")):
        vals = []
        for g in ("Control", "AD"):
            v = oof.loc[oof.group == g, col].values
            vals.append(v)
            boxstrip(ax, v, pos, GROUP_COLOR[g], width=0.40, seed=pos, size=3)
            ticks.append(pos); labels.append(g)
            pos += 1
        pg = S[f"recon_group_{tag}_mannwhitney_p"]
        hi = max(v.max() for v in vals)           # V7: clear every point, not the 97th pct
        bracket(ax, pos - 2, pos - 1, hi * 1.06, f"{stars(pg)}  {pmath(pg)}",
                tick=hi * 0.03, fontsize=6.6)
        ax.text(pos - 1.5, -0.16, nm, transform=ax.get_xaxis_transform(),
                ha="center", fontsize=7.6, color=INK)
        pos += 0.7
    ax.set_xticks(ticks); ax.set_xticklabels(labels, fontsize=6.8)
    ax.set_ylabel("Per-subject MSE"); ygrid(ax)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.14)
    # the subtitle is read off the test, not asserted: the CVAE arm is not
    # diagnosis-blind, and saying so would contradict the panel it labels
    verdict = ", ".join(
        f"{nm} {stars(S[f'recon_group_{tag}_mannwhitney_p'])}"
        for nm, tag in (("PLS-DA", "pls"), ("CVAE", "cvaetrue")))
    panel(ax, "d", "Does error depend on diagnosis?",
          f"Mann-Whitney, held-out subjects: {verdict}", dx=-0.17)

    save(fig, "fig6_reconstruction")



fig_reconstruction(S)

In [ ]:
# =====================================================================
# Figure 7 -- out-of-fold uncertainty
# =====================================================================
def fig_uncertainty(S):
    unc = load("oof_uncertainty.csv")
    panels = [("kl_true", "KL divergence (nats)", "KL — label-conditioned"),
              ("kl_neutral", "KL divergence (nats)", "KL — label-agnostic"),
              ("meanvar_true", "Mean latent variance", "Variance — label-conditioned"),
              ("meanvar_neutral", "Mean latent variance", "Variance — label-agnostic")]

    fig, axes = plt.subplots(2, 2, figsize=(TW, 6.2))
    axes = axes.ravel()
    for i, (col, ylab, title) in enumerate(panels):
        ax = axes[i]
        vals = []
        for j, g in enumerate(("Control", "AD")):
            v = unc.loc[unc.group == g, col].values
            vals.append(v)
            boxstrip(ax, v, j, GROUP_COLOR[g], seed=j, size=3.5)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Control", "AD"])
        ax.set_ylabel(ylab if i % 2 == 0 else "")
        ygrid(ax)
        p = S[f"{col}_mannwhitney_p"]
        hi = max(np.percentile(v, 99) for v in vals)
        lo = min(np.percentile(v, 1) for v in vals)
        span = hi - lo
        bracket(ax, 0, 1, hi + 0.12 * span, f"{stars(p)}  {pmath(p)}",
                tick=span * 0.05, fontsize=6.8)
        ax.set_ylim(lo - 0.08 * span, hi + 0.34 * span)
        delta = S[f"{col}_pct_increase_AD"]
        ax.text(0.02, 0.96, f"AD {delta:+.1f}% vs control", transform=ax.transAxes,
                ha="left", va="top", fontsize=6.8, color=INK2)
        panel(ax, "abcd"[i], title, dx=-0.24)
    fig.text(0.5, -0.035, "Held-out subjects only, pooled over the 20 splits "
                         f"($n = {int(S['unc_n_AD']):,}$ AD, "
                         f"{int(S['unc_n_control']):,} control observations).",
             ha="center", fontsize=7, color=INK2, style="italic")
    fig.tight_layout(w_pad=2.6, h_pad=3.0)
    save(fig, "fig7_uncertainty")



fig_uncertainty(S)

In [ ]:
# =====================================================================
# Figure 8 -- gene discovery and its discordance
# =====================================================================
def fig_genes(S):
    pls = load("PLSDA_gene_significance.csv")
    cv = load("CVAE_gene_significance.csv")
    de = load("gene_de_stats.csv")

    ps = pls[pls.significant].sort_values("mean_score", ascending=False)
    cs = cv[cv.significant].sort_values("mean_score", ascending=False)
    pls_set, cv_set = set(ps.gene), set(cs.gene)
    both = pls_set & cv_set

    # V7: the list panels grow with the longer gene list (0.105 in per row) so
    # 50-80 names stay legible; the Venn and the volcano keep their own rows.
    n_rows = max(len(ps), len(cs))
    list_h = max(3.6, 0.105 * n_rows)
    venn_h, volc_h = 2.4, 5.2
    fig = plt.figure(figsize=(TW, list_h + venn_h + volc_h + 1.6))
    gs = fig.add_gridspec(3, 2, width_ratios=[1.0, 1.0],
                          height_ratios=[list_h, venn_h, volc_h], hspace=0.30, wspace=0.52)

    def lollipop(ax, d, xlabel, letter, title, legend_loc="lower left"):
        """A dot on a stem, not a filled bar: 45 rows of heavy fill is a wall of
        ink, and the value is carried by one position either way."""
        d = d.iloc[::-1]
        y = np.arange(len(d))
        colors = [UP if x == "up" else DOWN if x == "down" else MID for x in d.direction]
        ax.hlines(y, 0, d.mean_score, color=RULE, lw=0.7, zorder=1)
        ax.scatter(d.mean_score, y, s=16, color=colors, zorder=3,
                   edgecolor=SURFACE, linewidth=0.5)
        italic_ticks(ax, list(d.gene), 5.6)
        ax.set_xlabel(xlabel)
        ax.set_xlim(0, d.mean_score.max() * 1.08)
        ax.set_ylim(-0.9, len(d) - 0.1)
        ax.tick_params(axis="y", length=0)
        xgrid(ax)
        despine(ax, left=False)
        panel(ax, letter, title, dx=-0.30)
        # inside the axes: the empty left margin of a lollipop chart is free
        # space, and a legend hung below collides with whatever panel follows
        ax.legend(handles=swatches([("up in AD", UP), ("down in AD", DOWN),
                                    ("no direction", MID)], marker="o", size=4),
                  loc=legend_loc, fontsize=6.3, framealpha=0.9, frameon=True,
                  facecolor=SURFACE, edgecolor="none", borderpad=0.4)

    lollipop(fig.add_subplot(gs[0, 0]), ps, "Mean VIP score", "a",
             f"PLS-DA — {len(ps)} stable genes")
    if len(cs):
        lollipop(fig.add_subplot(gs[0, 1]), cs, "Mean $|$IG$|$ attribution", "b",
                 f"CVAE — {len(cs)} stable genes")

    # (c) overlap -- Venn on the left, the shared genes listed on the right
    ax = fig.add_subplot(gs[1, 0])
    ax.set_aspect("equal"); ax.axis("off")
    ax.add_patch(Circle((-0.40, 0), 0.80, facecolor=PLS, alpha=0.22,
                        edgecolor=PLS, lw=1.2))
    ax.add_patch(Circle((0.40, 0), 0.80, facecolor=CVAE_T, alpha=0.22,
                        edgecolor=CVAE_T, lw=1.2))
    ax.text(-0.82, 0, f"{len(pls_set - cv_set)}", ha="center", va="center",
            fontsize=16, color=INK)
    ax.text(0.82, 0, f"{len(cv_set - pls_set)}", ha="center", va="center",
            fontsize=16, color=INK)
    ax.text(0, 0, f"{len(both)}", ha="center", va="center", fontsize=13, color=INK)
    ax.text(-0.82, 0.95, "PLS-DA only", ha="center", fontsize=7.4, color=PLS)
    ax.text(0.82, 0.95, "CVAE only", ha="center", fontsize=7.4, color=CVAE_T)
    ax.set_xlim(-1.75, 1.75); ax.set_ylim(-1.35, 1.35)
    panel(ax, "c", "Two gene sets, almost disjoint", dx=-0.16)
    axl = fig.add_subplot(gs[1, 1]); axl.axis("off")
    names = sorted(both)
    ncol = 1 if len(names) <= 6 else 2 if len(names) <= 16 else 3
    per = int(np.ceil(len(names) / ncol)) if names else 1
    axl.text(0.0, 0.98, f"Found by both ({len(names)})", transform=axl.transAxes,
             ha="left", va="top", fontsize=7.4, color=ACCENT)
    for c in range(ncol):
        chunk = names[c * per:(c + 1) * per]
        axl.text(0.0 + c * 0.36, 0.84, "\n".join(chunk) if chunk else "",
                 transform=axl.transAxes, ha="left", va="top", fontsize=6.2,
                 color=INK, fontstyle="italic", linespacing=1.25)
    axl.text(0.0, -0.02, f"Jaccard $=$ {S['jaccard']:.3f}\ngenome-wide "
                         f"$\\rho_\\mathrm{{Spearman}} =$ {S['spearman_rho_vip_ig']:.3f}",
             transform=axl.transAxes, ha="left", va="top", fontsize=7.0, color=INK2)

    # (d) concordance volcano
    ax = fig.add_subplot(gs[2, :])
    de = de.copy()
    de["nlp"] = -np.log10(de.DE_pvalue.clip(lower=1e-300))
    cls = np.full(len(de), "ns", dtype=object)
    gi = {g: i for i, g in enumerate(de.gene)}
    for g in pls_set - cv_set: cls[gi[g]] = "pls"
    for g in cv_set - pls_set: cls[gi[g]] = "cvae"
    for g in both:             cls[gi[g]] = "both"
    de["cls"] = cls
    style = {"ns":   (INK3,   1.6, 0.16, "o", f"not selected ({(cls == 'ns').sum():,})"),
             "pls":  (PLS,     16, 0.92, "o", f"PLS-DA only ({(cls == 'pls').sum()})"),
             "cvae": (CVAE_T,  16, 0.92, "s", f"CVAE only ({(cls == 'cvae').sum()})"),
             "both": (ACCENT,  38, 1.00, "D", f"both ({(cls == 'both').sum()})")}
    for k in ("ns", "pls", "cvae", "both"):
        s = de[de.cls == k]
        c, sz, al, mk, _ = style[k]
        ax.scatter(s.logFC, s.nlp, s=sz, color=c, alpha=al, marker=mk, linewidth=0.35,
                   edgecolor=SURFACE if k != "ns" else "none",
                   zorder=2 if k != "ns" else 1, rasterized=(k == "ns"))
    ax.axvline(0, color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
    ax.set_xlabel("Standardised expression difference (AD $-$ control)")
    ax.set_ylabel("$-\\log_{10}$ DE $p$-value")
    ygrid(ax)
    ax.legend(handles=[Line2D([], [], marker=style[k][3], ls="", color=style[k][0],
                              markersize=4.2, label=style[k][4])
                       for k in ("ns", "pls", "cvae", "both")],
              loc="upper left", ncol=1, fontsize=6.4, framealpha=0.92,
              frameon=True, facecolor=SURFACE, edgecolor="none")
    # every selected gene is named, not only the overlap: the point of the panel
    # is which genes each method picked, and an unlabelled mark cannot answer it.
    # Labels are laid out by `repel_labels`, which guarantees no two collide.
    sel = de[de.cls != "ns"]
    ax.set_xlim(*np.array(ax.get_xlim()) * 1.22)
    _top = ax.get_ylim()[1] * 1.30
    ax.set_ylim(-0.10 * _top, _top)          # V7: room below the floor for labels
    fig.canvas.draw()          # labels are placed in display space, so fix it first
    repel_labels(ax, sel.logFC.values, sel.nlp.values, list(sel.gene),
                 fontsize=5.0)
    panel(ax, "d", "Where the selected genes sit genome-wide",
          "selection is not simply the strongest differential expression", dx=-0.075)

    save(fig, "fig8_gene_discovery")



fig_genes(S)

In [ ]:
# =====================================================================
# Figure 9 -- directional expression profile of every selected gene
# =====================================================================
def fig_profile():
    pls = load("PLSDA_gene_significance.csv")
    cv = load("CVAE_gene_significance.csv")
    de = load("gene_de_stats.csv").set_index("gene")

    pls_set = set(pls.loc[pls.significant, "gene"])
    cv_set = set(cv.loc[cv.significant, "gene"])
    both = pls_set & cv_set
    groups = [("PLS-DA only", sorted(pls_set - cv_set), PLS),
              ("Found by both", sorted(both), ACCENT),
              ("CVAE only", sorted(cv_set - pls_set), CVAE_T)]
    genes = [g for _, gs_, _ in groups for g in gs_]
    n = len(genes)
    step = 2 * np.pi / n
    ang = np.arange(n) * step

    fig = plt.figure(figsize=(TW, TW))
    ax = fig.add_subplot(projection="polar")
    ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)

    ad = np.array([de.loc[g, "mean_z_AD"] for g in genes])
    ct = np.array([de.loc[g, "mean_z_control"] for g in genes])
    r0 = max(np.abs(np.concatenate([ad, ct]))) * 1.32   # radius of the zero ring
    rmax = r0 * 1.72

    # provenance blocks: a faint wedge plus a solid arc at the rim, so a gene
    # label can be traced back to the model that selected it without a legend
    start_i = 0
    for name, gs_, color in groups:
        if not gs_:
            continue
        a0, a1 = start_i * step - step / 2, (start_i + len(gs_)) * step - step / 2
        arc = np.linspace(a0, a1, 120)
        ax.fill_between(arc, 0, rmax, color=color, alpha=0.05, lw=0, zorder=0)
        ax.plot(arc, np.full_like(arc, rmax * 0.985), color=color, lw=2.4,
                solid_capstyle="butt", zorder=5)
        mid = (a0 + a1) / 2
        rot = np.degrees(-mid) % 360
        rot = rot - 180 if 90 < rot < 270 else rot
        ax.text(mid, rmax * (1.40 if n > 60 else 1.13), f"{name} ({len(gs_)})",
                ha="center", va="center", fontsize=8.0, color=color)
        start_i += len(gs_)

    close = lambda v: np.r_[v, v[:1]]
    ax.plot(close(ang), close(ct) + r0, color=CTRL, lw=1.2, zorder=3)
    ax.plot(close(ang), close(ad) + r0, color=AD, lw=1.2, ls=(0, (3.5, 2.0)), zorder=3)
    ax.scatter(ang, ct + r0, s=8, color=CTRL, zorder=4, edgecolor=SURFACE, linewidth=0.35)
    ax.scatter(ang, ad + r0, s=8, color=AD, zorder=4, marker="^",
               edgecolor=SURFACE, linewidth=0.35)
    ring = np.linspace(0, 2 * np.pi, 240)
    ax.plot(ring, np.full_like(ring, r0), color=INK2, lw=0.8, ls=(0, (2.5, 2.0)),
            zorder=2)

    ax.set_ylim(0, rmax)
    ax.set_yticks([r0 - 0.5, r0, r0 + 0.5])
    ax.set_yticklabels(["$-0.5$", "0", "$+0.5$"], fontsize=6.0, color=INK3)
    ax.set_rlabel_position(np.degrees(ang[0]) - 4)
    ax.set_xticks(ang)
    if n > 60:
        # V7: too many genes for horizontal labels -- write each one along its
        # own spoke so every name gets its own angular slot
        ax.set_position([0.15, 0.15, 0.70, 0.70])
        ax.set_xticklabels([])
        if r0 + 0.5 > 0.90 * rmax:      # the +0.5 ring would sit on the gene names
            ax.set_yticks([r0 - 0.5, r0]); ax.set_yticklabels(["$-0.5$", "0"], fontsize=6.0, color=INK3)
        for a, g in zip(ang, genes):
            phi = (90.0 - np.degrees(a)) % 360          # on-screen angle of the spoke
            flip = 90 < phi < 270
            ax.text(a, rmax * 1.02, g, rotation=phi - 180 if flip else phi,
                    rotation_mode="anchor", ha="right" if flip else "left",
                    va="center", fontsize=5.0, fontstyle="italic", color=INK,
                    clip_on=False)
    else:
        ax.set_xticklabels(genes, fontsize=5.6, fontstyle="italic")
        ax.tick_params(axis="x", pad=-1)
    ax.grid(color=GRID, lw=0.45)
    ax.spines["polar"].set_visible(False)
    ax.legend(handles=[Line2D([], [], color=CTRL, lw=1.4, marker="o", markersize=4,
                              label="Control"),
                       Line2D([], [], color=AD, lw=1.4, ls=(0, (3.5, 2.0)),
                              marker="^", markersize=4, label="AD")],
              loc="center", bbox_to_anchor=(0.5, 0.5), fontsize=8, labelspacing=0.5,
              handlelength=2.2)
    fig.text(0.5, 0.045,
             f"Mean $z$-score by diagnosis for all {n} genes selected by either model. "
             "The dashed ring\nis the cohort mean; outside it the gene is elevated, "
             "inside it reduced.",
             ha="center", fontsize=7.2, color=INK2, style="italic")
    save(fig, "fig9_expression_profile")



fig_profile()

In [ ]:
# =====================================================================
# Figure 10 -- how each latent dimension maps back onto genes
# =====================================================================
def fig_latent_map():
    pls = load("PLSDA_gene_significance.csv")
    cv = load("CVAE_gene_significance.csv")
    snr = load("latent_snr.csv")
    genes = list(load("gene_de_stats.csv").gene)
    gidx = {g: i for i, g in enumerate(genes)}

    pls_set = set(pls.loc[pls.significant, "gene"])
    cv_set = set(cv.loc[cv.significant, "gene"])
    blocks = [(sorted(pls_set - cv_set), PLS), (sorted(pls_set & cv_set), ACCENT),
              (sorted(cv_set - pls_set), CVAE_T)]
    sel = [g for b, _ in blocks for g in b]
    cols = [gidx[g] for g in sel]

    # V7: more than 70 genes do not fit one row of names, so each map is drawn
    # in two stacked halves with the same colour scale
    halves = 1 if len(sel) <= 70 else 2
    per = int(np.ceil(len(sel) / halves))
    color_of = {g: c for b, c in blocks for g in b}
    fig = plt.figure(figsize=(TW, 8.0 + 3.4 * (halves - 1)))
    gs = fig.add_gridspec(2, 1, height_ratios=[1.55 * halves, 1.0], hspace=0.30 if halves == 1 else 0.24)
    gs_top = gs[0].subgridspec(2 * halves, 1, hspace=0.80)
    gs_bot = gs[1].subgridspec(1, 3, wspace=0.30)

    for i, (tag, name, prefix, cmap) in enumerate(
            (("PLS-DA", "PLS-DA", "LV", SEQ_PLS),
             ("CVAE_true", "CVAE (label-conditioned)", "Z", SEQ_CVAE))):
        Wall = np.abs(np.load(DATA / f"latent_gene_weights_{tag}.npy"))[:, cols]
        # scaled by the model's own global maximum, not row by row: whether one
        # dimension carries more weight than another is exactly the comparison
        Wall = Wall / (Wall.max() + 1e-12)
        for h in range(halves):
            lo, hi = h * per, min((h + 1) * per, len(sel))
            W, sel_h = Wall[:, lo:hi], sel[lo:hi]
            ax = fig.add_subplot(gs_top[i * halves + h])
            im = ax.imshow(W, aspect="auto", cmap=cmap, vmin=0, vmax=1,
                           interpolation="nearest")
            ax.set_xlim(-0.5, per - 0.5)               # same column width in both halves
            ax.set_yticks(range(W.shape[0]))
            ax.set_yticklabels([f"{prefix}{k+1}" for k in range(W.shape[0])], fontsize=6.4)
            italic_ticks(ax, sel_h, 4.6, axis="x")
            ax.tick_params(axis="x", rotation=90, length=0)
            ax.tick_params(axis="y", length=0)
            for sp in ax.spines.values():
                sp.set_visible(False)
            # provenance rule above the map, one segment per run of same-source genes
            x0 = 0
            while x0 < len(sel_h):
                c = color_of[sel_h[x0]]; x1 = x0
                while x1 < len(sel_h) and color_of[sel_h[x1]] == c:
                    x1 += 1
                ax.plot([x0 - 0.5, x1 - 0.5], [-0.72] * 2, color=c, lw=2.6,
                        clip_on=False, solid_capstyle="butt")
                x0 = x1
            cb = fig.colorbar(im, ax=ax, fraction=0.014, pad=0.010)
            cb.set_label("relative $|$weight$|$", fontsize=6.2, color=INK2)
            cb.ax.tick_params(labelsize=5.8, length=2); cb.outline.set_visible(False)
            if h == 0:
                sub = "scaled by this model's global maximum" + (
                    f"; genes {lo+1}-{hi} of {len(sel)}, continued below" if halves > 1 else "")
                panel(ax, "ab"[i], name, sub, dx=-0.045)
    fig.legend(handles=swatches([("PLS-DA only", PLS), ("both", ACCENT),
                                 ("CVAE only", CVAE_T)]),
               title="gene provenance (rule above each map)", title_fontsize=6.4,
               alignment="right", loc="upper right", bbox_to_anchor=(0.995, 1.002),
               ncol=3, fontsize=6.8)

    # ---- signal-to-noise per latent dimension ------------------------
    ymax = snr.snr.max() * 1.22
    for i, (tag, color, name) in enumerate((("PLS-DA", PLS, "PLS-DA latent variables"),
                                            ("CVAE_true", CVAE_T, "CVAE (label-conditioned)"),
                                            ("CVAE_neutral", CVAE_N, "CVAE (label-agnostic)"))):
        ax = fig.add_subplot(gs_bot[i])
        d = snr[snr.model == tag]
        x = np.arange(len(d))
        ax.bar(x, d.snr, 0.62, color=color, edgecolor=SURFACE, linewidth=1.0)
        ax.axhline(1.0, color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
        ax.set_xticks(x); ax.set_xticklabels(d.latent, fontsize=6.6)
        ax.set_ylim(0, ymax); ygrid(ax)
        ax.set_ylabel("Signal-to-noise" if i == 0 else "")
        # a "best dimension" label would be meaningless where the bars are level,
        # so report the spread instead and let the reader see which case it is
        panel(ax, "cde"[i], name,
              f"range {d.snr.min():.2f}–{d.snr.max():.2f}, "
              f"fold {d.snr.max()/max(d.snr.min(), 1e-9):.1f}$\\times$", dx=-0.16)
    fig.text(0.5, -0.025, "Signal-to-noise is the mean absolute weight of the selected "
                         "genes divided by that of the remaining transcriptome; the "
                         "dashed line is parity.",
             ha="center", fontsize=7, color=INK2, style="italic")
    save(fig, "fig10_latent_gene_map")



fig_latent_map()

In [ ]:
# =====================================================================
# Figure 11 -- functional enrichment
# =====================================================================
def fig_enrichment():
    files = [("plsda_up", "PLS-DA, up in AD"), ("plsda_down", "PLS-DA, down in AD"),
             ("cvae_up", "CVAE, up in AD"), ("cvae_down", "CVAE, down in AD")]
    TOPN = 20
    fig, axes = plt.subplots(2, 2, figsize=(TW, 12.6))
    # one x-scale across the four sets: the CVAE-up panel reaching twice as far
    # as the others is the finding, and per-panel scales would hide it
    xmax = max((pd.read_csv(DATA / "enrichment" / f"{n}.csv")
                  .negative_log10_of_adjusted_p_value.max())
               for n, _ in files
               if len(pd.read_csv(DATA / "enrichment" / f"{n}.csv"))) * 1.14
    for ax, (name, title) in zip(axes.ravel(), files):
        d = pd.read_csv(DATA / "enrichment" / f"{name}.csv")
        if not len(d):
            ax.axis("off")
            panel(ax, "", f"{title} — no enriched terms")
            continue
        # g:Profiler returns one row per database identifier, so the same complex
        # can appear several times under one name -- keep the strongest.
        n_all = len(d)
        d = d.sort_values("adjusted_p_value").drop_duplicates(subset=["source", "term_name"])
        n_uniq = len(d)
        d = d.nsmallest(min(TOPN, len(d)), "adjusted_p_value").iloc[::-1]
        y = np.arange(len(d))
        colors = [SRC_COLOR.get(s, MID) for s in d.source]
        single = (d.intersection_size.values == 1)
        ax.hlines(y, 0, d.negative_log10_of_adjusted_p_value, color=RULE, lw=0.7, zorder=1)
        ax.scatter(d.negative_log10_of_adjusted_p_value, y, s=17, zorder=3,
                   facecolor=[SURFACE if s1 else c for c, s1 in zip(colors, single)],
                   edgecolor=[c if s1 else SURFACE for c, s1 in zip(colors, single)],
                   linewidth=[0.9 if s1 else 0.5 for s1 in single])
        labels = [(t[:42] + "…") if len(t) > 43 else t for t in d.term_name]
        ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=5.9)
        for tick, s1 in zip(ax.get_yticklabels(), single):
            tick.set_color(INK3 if s1 else INK)
        for yi, (v, k) in enumerate(zip(d.negative_log10_of_adjusted_p_value,
                                        d.intersection_size)):
            ax.text(v + 0.032 * xmax, yi, f"({int(k)})" if k == 1 else f"{int(k)}",
                    va="center", fontsize=5.4,
                    color=INK3 if k == 1 else INK2)
        ax.axvline(-np.log10(0.05), color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
        ax.set_xlabel("$-\\log_{10}$ FDR-adjusted $p$")
        ax.set_xlim(0, xmax)
        ax.set_ylim(-0.8, len(d) - 0.2)
        ax.tick_params(axis="y", length=0)
        xgrid(ax); despine(ax, left=False)
        dup = f", {n_all - n_uniq} duplicate ids merged" if n_all != n_uniq else ""
        shown = f"top {len(d)} of {n_uniq}" if n_uniq > len(d) else f"all {n_uniq}"
        used = list(dict.fromkeys(d.source))
        n1 = int((d.intersection_size == 1).sum())
        note = f"; {n1} of {len(d)} rest on one gene" if n1 else ""
        panel(ax, "", title, f"{shown} terms{dup}" + note, dx=-0.42)
        ax.legend(handles=swatches([(s, SRC_COLOR.get(s, MID)) for s in used], marker="o",
                                   size=4),
                  loc="upper center", bbox_to_anchor=(0.5, -0.11),
                  ncol=min(len(used), 6), fontsize=6.2)
    for ax, l in zip(axes.ravel(), "abcd"):
        ax.annotate(l, xy=(-0.46, 1.0), xycoords="axes fraction", fontsize=9.5,
                    fontweight="bold", family="DejaVu Sans", color=INK,
                    ha="left", va="bottom", annotation_clip=False)
    fig.tight_layout(w_pad=8.0, h_pad=5.6)
    # the convention, stated once for all four panels
    fig.text(0.5, -0.012,
             "Trailing number = query genes in the term. Grey label, hollow dot "
             "and a bracketed count mark a term whose enrichment rests on a single "
             "gene; those terms are reported for completeness and are not interpreted.",
             ha="center", va="top", fontsize=6.6, color=INK2, style="italic")
    save(fig, "fig11_enrichment")



fig_enrichment()

In [ ]:
# =====================================================================
# What the enrichment can and cannot support, in numbers rather than adjectives
# =====================================================================
# Printed rather than typed into the manuscript, so the sentences the paper
# makes about single-gene terms cannot drift from the run that produced them.
from collections import Counter

print("Background: {:,} genes; every set tested against the same domain.\n"
      .format(len(load("gene_de_stats.csv"))))

for name, title in [("plsda_up", "PLS-DA, up in AD"), ("plsda_down", "PLS-DA, down in AD"),
                    ("cvae_up", "CVAE, up in AD"), ("cvae_down", "CVAE, down in AD")]:
    d = pd.read_csv(DATA / "enrichment" / f"{name}.csv")
    q = len(pd.read_csv(DATA / f"{'PLSDA' if name.startswith('plsda') else 'CVAE'}"
                        f"_{'upregulated' if name.endswith('up') else 'downregulated'}"
                        "_genes.csv"))
    if not len(d):
        print(f"{title}: {q} genes -> no enriched terms")
        continue
    u = d.sort_values("adjusted_p_value").drop_duplicates(subset=["source", "term_name"])
    n1 = int((u.intersection_size == 1).sum())
    # which gene is doing the work in the single-gene terms
    drivers = Counter(g for gs in u.loc[u.intersection_size == 1, "intersections"].fillna("")
                      for g in str(gs).split(",") if g)
    top = ", ".join(f"{g} ({c})" for g, c in drivers.most_common(3))
    print(f"{title}: {q} genes -> {len(d)} terms ({len(u)} unique), "
          f"{n1} rest on a single gene, {len(u) - n1} on two or more.")
    if top:
        print(f"    single-gene terms are driven mainly by: {top}")
print("\nOnly terms with two or more query genes are interpreted in the text.")


---

### Reading the enrichment figure

Enrichment was run on small gene sets (sizes printed above) against the same merged gene
background used everywhere else in this notebook, with FDR control at
*q* < 0.05. At that query size the test is reachable only for small terms: a
20-gene query can contribute at most 20 hits, so a Gene Ontology term with
several hundred members can never clear the threshold, while a curated complex
with two or three members clears it on a single hit. **The number of terms a
panel shows is therefore a statement about how densely its genes are annotated,
not about how much biology they carry, and the four panels are not comparable
on that axis.**

Terms supported by a single gene are reported for completeness -- they are
correct hypergeometric results -- but they are not interpreted, and the figure
marks them (grey label, hollow dot, bracketed count). They carry one gene's
annotation rather than a pattern: drop that gene and the term disappears, and a
single well-curated gene pulls in every complex it belongs to at once. The cell
above prints, for each set, how many terms rest on one gene and which genes
those are.

The cell above prints, per set, how many terms rest on a single gene and which
gene carries them, so these statements are read off this run rather than typed.


---

# Output manifest

Everything this notebook produced, with sizes. A zero-byte file means a figure
failed to render, so it is called out rather than left to be noticed later.

In [ ]:
# every artefact this notebook wrote, as a single checkable manifest
import os

rows = []
for root, _, files in os.walk(OUT):
    for f in sorted(files):
        if f.startswith("__") or f.endswith(".ipynb"):
            continue
        p = os.path.join(root, f)
        rows.append({"file": os.path.relpath(p, OUT),
                     "bytes": os.path.getsize(p)})

manifest = pd.DataFrame(rows).sort_values("file").reset_index(drop=True)
manifest.to_csv(f"{OUT}/output_manifest.csv", index=False)
print(manifest.to_string(index=False))

empty = manifest[manifest.bytes == 0]
print(f"\n{len(manifest)} files, {manifest.bytes.sum()/1e6:.1f} MB")
if len(empty):
    print("EMPTY FILES -- something failed to render:")
    print(empty.to_string(index=False))
else:
    print("no empty files")